# Sesión 6 · RAG intermedio: chunking y recuperación

**AI Engineer I: Fundamentos de IA Generativa** · Módulo 1 · Edutek

---

## 🎯 Objetivos de la sesión

Al terminar este notebook podrás:

1. **Ingerir documentos reales** (Markdown, HTML, DOCX y PDF), conservando su estructura, y explicar cuándo hace falta **OCR**. Y conocer un parser moderno con análisis de *layout* (**Docling**: tablas y OCR).
2. **Comparar estrategias de fragmentado** (*chunking*) **con números**: tamaño fijo, oraciones, recursivo, semántico, por sección y padre-hijo (y el chunking híbrido de Docling).
3. **Combinar búsqueda léxica (BM25) y semántica** con fusión **RRF** (búsqueda híbrida), y saber cuándo cada una falla.
4. **Filtrar por metadatos**, **reordenar candidatos (reranking)** y **reescribir la consulta** cuando la pregunta es parte de una conversación.
5. **Evaluar la recuperación** con *hit@k* y *MRR*, y leer los resultados con criterio (ruido, costo de contexto).

> ⚠️ **Sigue sin ser un agente.** El flujo continúa siendo fijo: nadie decide qué herramienta usar. Lo que mejora es **la calidad de lo que llega al prompt**.

## 🗺️ Mapa del notebook

| Parte | Tema | ¿Consume dinero? |
| --- | --- | --- |
| 0 | Preparar el entorno | No |
| 1 | De un RAG simple a uno intermedio: dónde falla y cómo medirlo | No (embeddings locales) |
| 2 | Ingesta multiformato: HTML, DOCX, PDF, OCR y Docling (opcional) | No |
| 3 | Estrategias de chunking (incluye el híbrido de Docling) | No (embeddings locales) |
| 4 | **Laboratorio de chunking** (con padre-hijo y Docling) | No (embeddings locales) |
| 5 | Búsqueda léxica: BM25 | No |
| 6 | Búsqueda híbrida y fusión RRF | No |
| 7 | Metadatos y filtros | No |
| 8 | Reranking | **Sí, centavos** (o gratis con modelo local) |
| 9 | Reescritura de la consulta y multi-consulta | **Sí, centavos** (o gratis con modelo local) |
| 10 | El pipeline completo y la práctica | **Sí, centavos** (opcional) |
| 11 | Cierre | No |

> 💰 Las celdas que llaman al LLM llevan **💰 FYI** y una estimación. Con Anthropic u OpenAI, toda la sesión cuesta **menos de US$ 0.02**; con un modelo local en LM Studio, cuesta 0.

> 📦 **Docling es opcional** (2.5, 3.4 y 4.2): instala el grupo con `uv sync --group docling` **antes de clase**; descarga varios cientos de MB de modelos la primera vez. Sin él, esas celdas avisan y el resto funciona.

---
# Parte 0 · Preparar el entorno

In [ ]:
import sys

print("Python:", sys.version.split()[0])
assert sys.version_info[:2] == (3, 12), (
    "Esta sesión está fijada a Python 3.12. Ejecuta `uv sync` dentro de la carpeta "
    "s06-rag-intermedio y selecciona el kernel .venv (ver README)."
)

In [ ]:
import json
import os
import time
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

from cliente_llm import Gasto, hay_api_key, local_disponible, modelo_local, variable_api_key
from embeddings import embeddings_disponibles, modelo_embeddings, proveedor_embeddings
from fragmentado import ESTRATEGIAS, fragmentar_corpus, fragmentar_doc, fragmentar_padre_hijo, resumen_fragmentos
from ingesta import leer_carpeta, leer_documento_multi
from rag import proveedor_generacion, proveedores_disponibles
from recuperacion import (
    BM25, MODOS, buscar_bm25, buscar_hibrido, buscar_multiconsulta, buscar_vectorial, construir_indice,
    contiene_clave, evaluar_busqueda, expandir_a_padre, expandir_consulta, fusion_rrf, reescribir_consulta,
    rerank_llm, responder_avanzado, tokenizar,
)

HAY_EMB, detalle = embeddings_disponibles()
print(("✅ Embeddings: " if HAY_EMB else "❌ Embeddings: ") + detalle)

### 🔌 Quién ejecuta los pasos con LLM

Tres técnicas de esta sesión (reranking, reescritura de la consulta y multi-consulta) **le piden ayuda a un LLM**, además de la generación de la respuesta. Se elige con `RAG_PROVIDER` (o, si no existe, `LLM_PROVIDER`) en tu `.env`: `anthropic`, `openai` o `local` (LM Studio).

> 🔎 **Una advertencia que verás medida:** un modelo local de 3B parámetros sirve para ver el flujo, pero hace estas tareas peor que un modelo de proveedor. Es parte de la lección: **cada técnica que añade un LLM debe ganarse su lugar con una medición**.

In [ ]:
GEN = proveedor_generacion()
if GEN == "local":
    HAY_LLM, msg = local_disponible(modelo_local())
else:
    HAY_LLM, msg = hay_api_key(GEN), f"falta {variable_api_key(GEN)} en tu .env"
print(f"LLM para los pasos con IA: {GEN} · {'✅ disponible' if HAY_LLM else '❌ ' + str(msg)}")

### 🔌 Interruptor general de gasto

💰 **FYI:** con `False`, ninguna celda llama al LLM (las partes 8, 9 y 10 se omiten). Te recomendamos un límite de gasto mensual en tu cuenta (ver `docs/COSTOS.md`).

In [ ]:
EJECUTAR_LLAMADAS_DE_PAGO = True   # ← ponla en False para no llamar al LLM
PUEDE_GASTAR = EJECUTAR_LLAMADAS_DE_PAGO and HAY_LLM and HAY_EMB
print("Celdas con LLM activas:", PUEDE_GASTAR)
gasto = Gasto()

---
# Parte 1 · De un RAG simple a uno intermedio

## Lo que tenías al terminar la Sesión 5

```
INGESTA     un .md  ─► fragmentar (sección / oraciones / fijo) ─► vectorizar ─► guardar
CONSULTA    pregunta ─► vectorizar ─► los k más parecidos ─► prompt ─► LLM ─► respuesta con citas
```

Funcionaba con un manual **ordenado**, con **una sola** forma de buscar. En la vida real aparecen cuatro problemas:

| Problema | Ejemplo | Técnica que lo ataca (Parte) |
| --- | --- | --- |
| Los documentos **no son Markdown** | Un contrato en Word, una guía en una página web, un boletín en PDF (a veces escaneado) | **Ingesta multiformato** (2) y parser con *layout* (2.5) |
| Los fragmentos **cortan o diluyen** las ideas | Una regla partida a la mitad; una sección entera de 200 palabras con 3 temas | **Estrategias de chunking** (3 y 4) |
| La búsqueda por significado **falla con lo exacto** | "¿Qué indica la alarma OPT-1107?": los códigos casi no tienen "significado" | **BM25 + búsqueda híbrida** (5 y 6) |
| La misma pregunta tiene **respuestas distintas según quién pregunta** | "¿En cuánto tiempo reparan una falla?" ≠ para un hogar y para una empresa | **Metadatos y filtros** (7) |
| La pregunta **no se entiende sola** | "¿Y los domingos?" | **Reescritura de la consulta** (9) |
| El fragmento correcto está **en el puesto 5**, pero solo enviamos 3 | | **Reranking** (8) |

## 🏗️ Analogía de infraestructura

Piensa en un **sistema de observabilidad**: una búsqueda por texto libre (como `grep`) encuentra el código exacto `OPT-1107`, pero no sabe que "caída de voltaje" y "voltaje bajo" son lo mismo; una búsqueda semántica sí lo sabe, pero se pierde con identificadores. Un buen sistema usa **las dos**, **filtra** por entorno o por equipo (`env=prod`), y deja que **un revisor** reordene los candidatos antes de actuar. Eso es exactamente un RAG intermedio.

## Qué corpus usaremos

Cuatro documentos **ficticios** de "Andes Fibra", en cuatro formatos, y **30 preguntas de prueba** con la respuesta esperada. Conviene medir **desde ya** para saber si cada mejora sirve.

In [ ]:
DATOS = Path("datos")
documentos = leer_carpeta(DATOS)
print(f"{'documento':<28}{'tipo':<6}{'secciones':>10}{'palabras':>10}")
for d in documentos:
    palabras = sum(len(t.split()) for _, t in d.secciones)
    print(f"{d.fuente:<28}{d.tipo:<6}{len(d.secciones):>10}{palabras:>10}")
    for aviso in d.avisos:
        print(f"    ⚠️ {aviso}")

Fíjate en el aviso de `acta_escaneada.pdf`: **no se pudo leer ningún texto**. Lo resolvemos en la Parte 2 (sección 2.5, con OCR).

## Las preguntas de prueba

Cada pregunta trae una **`clave`**: el texto que debe aparecer en un fragmento para que *ese fragmento* contenga la respuesta. Así podemos medir **cualquier** estrategia de fragmentado, aunque cambien los fragmentos (en la Sesión 5 comparábamos secciones; aquí los fragmentos cambian de forma y tamaño).

| Tipo | Qué pone a prueba | Ejemplo |
| --- | --- | --- |
| `semantica` | Paráfrasis: casi no comparte palabras con el documento | "Si me voy antes del año, ¿cuánto me cobran de multa?" |
| `lexica` | Un código, modelo o número **exacto** | "¿Qué indica la alarma OPT-1107?" |
| `tabla` | Datos que estaban en una tabla (DOCX/PDF) | "¿Qué día hacen el mantenimiento en la zona sur?" |
| `filtro` | La respuesta depende del **tipo de cliente** | "¿En cuánto tiempo deben reparar una falla?" |
| `conversacional` | Necesita el **historial** para entenderse | "¿Y los domingos?" |

In [ ]:
preguntas = json.loads(Path("datos/preguntas_recuperacion.json").read_text(encoding="utf-8"))
por_tipo = {}
for p in preguntas:
    por_tipo.setdefault(p["tipo"], []).append(p)
for tipo, lista in por_tipo.items():
    print(f"{tipo:<15}{len(lista):>3} preguntas  ·  ej.: {lista[0]['pregunta']}")

# Las preguntas "base": se pueden hacer sin filtro ni historial. Con ellas comparamos estrategias y buscadores.
BASE = [p for p in preguntas if p["tipo"] in ("semantica", "lexica", "tabla")]
print(f"\nPreguntas base (semantica + lexica + tabla): {len(BASE)}")

## ¿Cómo medimos? `hit@k`, `MRR` y el costo del contexto

| Métrica | Qué responde |
| --- | --- |
| **hit@1** | ¿En cuántas preguntas el **primer** fragmento contiene la respuesta? |
| **hit@3** | ¿Y entre los **tres** primeros? (los que enviaríamos al LLM) |
| **MRR** (*Mean Reciprocal Rank*) | Promedio de `1 / posición` del primer acierto. 1.0 = siempre primero; 0.5 = suele ser el segundo; 0 = nunca aparece. Premia **estar arriba**, no solo estar. |
| **contexto** | Palabras promedio de los k fragmentos: **es lo que pagas en tokens** al armar el prompt. |

> ⚠️ Esta medición solo mira la **recuperación**, no la respuesta del LLM (igual que en la Sesión 5: si el dato correcto no llega al prompt, ningún prompt lo arregla). Y con 22 preguntas, **1 o 2 aciertos de diferencia son ruido**.

### Las métricas, aterrizadas

**🧠 Qué es.** Tres números y un costo que resumen "qué tan bien busca" tu RAG, **sin llamar al LLM**. Se calculan sobre un conjunto de preguntas de prueba donde ya sabemos qué texto debe aparecer (la `clave`).

**⚙️ Qué hace cada una.**

| Métrica | Pregunta que responde | Cómo se calcula (por pregunta) |
| --- | --- | --- |
| **hit@1** | ¿La respuesta quedó **primera**? | 1 si el primer resultado contiene la `clave`; 0 si no |
| **hit@k** | ¿Está **entre los k primeros**? (los que enviaríamos al LLM) | 1 si algún resultado de los k contiene la `clave` |
| **MRR** | ¿Qué tan **arriba** quedó? | `1 / posición` del primer acierto; 0 si no aparece. Luego se promedia |
| **contexto** | ¿Cuánto texto envío al LLM? | Palabras de los k resultados, promediadas |

**🎯 Para qué sirve.** Para decidir con números en vez de con la impresión de haber probado dos ejemplos. Cada técnica de la sesión tiene que **mejorar estas métricas** (o justificar su costo).

**🏗️ Analogía de infraestructura.** Es un **conjunto de pruebas de regresión** para tu buscador: las preguntas son los casos de prueba y hit@k es el porcentaje que pasa. **MRR** es como medir en qué posición aparece el servicio correcto en una lista de resultados de monitoreo: no basta con que esté en la lista, importa que esté arriba, porque nadie revisa la posición 10. **El contexto** es el costo de ejecución de cada prueba.

**✏️ Ejemplo a mano (4 preguntas).** La respuesta correcta quedó en el puesto 1, 1, 2 y "no apareció":

| Pregunta | Posición del primer acierto | 1 ÷ posición |
| --- | --- | --- |
| 1 | 1.º | 1.00 |
| 2 | 1.º | 1.00 |
| 3 | 2.º | 0.50 |
| 4 | no apareció | 0.00 |

`hit@1` = 2/4 · `hit@3` = 3/4 · `MRR` = (1 + 1 + 0.5 + 0) ÷ 4 = **0.625**. El contexto, si en una pregunta los 3 fragmentos tenían 50, 60 y 40 palabras, es **150** palabras para esa pregunta; se promedia con las demás.

**⚠️ Cuándo no basta.**
- Miden **recuperación**, no si el LLM responde bien (eso viene en la Sesión 7).
- Con 22 preguntas, 1 o 2 aciertos de diferencia son **ruido**.
- hit@1 y MRR **premian a los fragmentos grandes** (un trozo grande casi siempre "contiene" la clave): por eso se mira siempre el **contexto** al lado.

**🐞 Errores típicos.** (1) Declarar ganador por 1 pregunta. (2) Comparar MRR sin mirar el contexto. (3) Cambiar **varias** cosas a la vez y no saber cuál mejoró. (4) Ajustar parámetros con las mismas preguntas con que evalúas (sobreajuste).

Prueba el cálculo a mano en la celda siguiente (no consume nada):

> 💡 El **MRR** es la métrica que más cuesta entender, así que tiene **su propia sección** justo después de esta celda: la receta en 4 pasos, por qué es `1 ÷ puesto`, para qué sirve si ya existe `hit@k`, y un mini ejercicio.

In [ ]:
# Ejemplo a mano: MRR con 4 preguntas de juguete (sin LLM, sin embeddings, sin costo)
posiciones = [1, 1, 2, None]          # dónde quedó el primer acierto de cada pregunta (None = no apareció)
rr = [1 / p if p else 0.0 for p in posiciones]
print("Rango recíproco por pregunta:", [round(x, 2) for x in rr])
print("MRR =", sum(rr) / len(rr))
print("hit@1 =", sum(p == 1 for p in posiciones), "/", len(posiciones),
      " · hit@3 =", sum(p is not None and p <= 3 for p in posiciones), "/", len(posiciones))

### 📐 MRR, con calma (la métrica que más confunde)

**La pregunta que responde.** *"Cuando busco, ¿qué tan arriba queda la respuesta correcta?"* No basta con que aparezca en la lista: importa **en qué puesto**.

**La receta, en 4 pasos** (se repite por cada pregunta de prueba):

1. Buscas y obtienes una **lista ordenada** de resultados (puesto 1, 2, 3…).
2. Miras en qué **puesto** está el **primer** resultado que contiene la respuesta.
3. Calculas el **rango recíproco**: `1 ÷ puesto`. Si la respuesta no aparece, vale **0**.
4. **Promedias** esos valores sobre todas las preguntas. Ese promedio es el **MRR** (*Mean Reciprocal Rank*, "rango recíproco medio").

| Puesto del primer acierto | `1 ÷ puesto` | Cómo se lee |
| --- | --- | --- |
| 1.º | **1.00** | perfecto |
| 2.º | **0.50** | vale la mitad |
| 3.º | **0.33** | un tercio |
| 4.º | **0.25** | un cuarto |
| 10.º | **0.10** | casi nadie mira hasta ahí |
| no apareció | **0** | no cuenta |

**¿Por qué `1 ÷ puesto` y no simplemente el puesto?** Tres razones:

1. **Premia estar arriba, y la caída es rápida.** Pasar del 1.º al 2.º cuesta **0.50**; pasar del 9.º al 10.º cuesta solo **0.01**. Refleja cómo se usa una lista de resultados: quien busca (y el LLM, que lee primero los fragmentos de arriba) presta atención a los primeros puestos.
2. **Resuelve el "no apareció".** Con el puesto promedio no sabrías qué número poner (¿infinito?). Con el recíproco, "nunca" es simplemente 0.
3. **Aguanta los casos extremos.** Una pregunta cuya respuesta quedó en el puesto 50 destruiría un promedio de puestos; en recíproco aporta 0.02 y no arruina nada.

**Cómo se lee el resultado final:**

| MRR | Significa |
| --- | --- |
| **1.0** | la respuesta correcta quedó **siempre primera** |
| **≈ 0.9** | casi siempre primera |
| **≈ 0.5** | suele quedar **segunda** |
| **0** | **nunca** aparece |

**🏗️ Analogía de infraestructura.** Buscas en la wiki "cómo reiniciar el balanceador". Si el runbook correcto es el **primer** resultado, lo abres y listo (1.0). Si es el **segundo**, ya abriste uno que no servía (0.5). Si es el **décimo**, perdiste la paciencia mucho antes (0.1). El MRR mide, en promedio, **cuánto cuesta encontrar la respuesta correcta** en tu buscador.

**✏️ Ejemplo a mano (el mismo de arriba).** Cuatro preguntas; la respuesta correcta quedó en el puesto **1, 1, 2** y **no apareció**:

`MRR = (1 + 1 + 0.5 + 0) ÷ 4 = 2.5 ÷ 4 = 0.625`

Míralo paso a paso, con tablita y suma acumulada (no consume nada):

In [ ]:
def mrr_paso_a_paso(posiciones, nombres=None):
    """Imprime, pregunta por pregunta, cómo se llega al MRR.

    `posiciones`: el puesto del primer acierto de cada pregunta (None = la respuesta no apareció).
    """
    total = 0.0
    print(f"{'pregunta':<10}{'puesto':>12}{'1 ÷ puesto':>12}{'suma acumulada':>16}")
    for i, p in enumerate(posiciones):
        rr = 1 / p if p else 0.0                       # el rango recíproco de ESTA pregunta
        total += rr
        etiqueta = nombres[i] if nombres else i + 1
        print(f"{etiqueta!s:<10}{(p if p else 'no apareció')!s:>12}{rr:>12.2f}{total:>16.3f}")
    mrr = total / len(posiciones)                       # el promedio sobre TODAS las preguntas
    print(f"\nMRR = {total:.3f} ÷ {len(posiciones)} preguntas = {mrr:.3f}")
    return mrr

_ = mrr_paso_a_paso([1, 1, 2, None])

#### ¿Para qué sirve el MRR si ya tenemos `hit@k`?

`hit@k` solo responde **sí o no** ("¿está entre los k primeros?"). **No distingue el 1.º del 3.º.** El MRR sí. Mira tres buscadores de juguete sobre las mismas 4 preguntas:

In [ ]:
# Tres buscadores de juguete: el puesto del primer acierto en cada una de las 4 preguntas
buscadores = {
    "A · siempre primero":         [1, 1, 1, 1],
    "B · siempre tercero":         [3, 3, 3, 3],
    "C · primero, pero falla una": [1, 1, 1, None],
}
print(f"{'buscador':<30}{'hit@1':>8}{'hit@3':>8}{'MRR':>8}")
for nombre, pos in buscadores.items():
    n = len(pos)
    hit1 = sum(p == 1 for p in pos) / n
    hit3 = sum(p is not None and p <= 3 for p in pos) / n
    mrr = sum(1 / p if p else 0.0 for p in pos) / n
    print(f"{nombre:<30}{hit1:>8.0%}{hit3:>8.0%}{mrr:>8.2f}")

**Qué ves:**

- **A y B tienen el mismo `hit@3` (100 %)**, pero B manda **siempre** la respuesta al tercer puesto. Solo el **MRR** (1.00 contra 0.33) y `hit@1` lo delatan.
- **C contra B es el giro:** `hit@3` dice que **B es mejor** (100 % contra 75 %); el MRR dice que **C es mejor** (0.75 contra 0.33). **No hay contradicción: miden cosas distintas.**
  - Si tu RAG envía los **3 primeros** fragmentos al LLM, lo que importa es que la respuesta **esté dentro** → mira `hit@3`.
  - Si solo mostrarías **un** resultado a una persona, o si el modelo se distrae con el ruido de los fragmentos de arriba, importa **qué tan arriba** → mira `hit@1` y **MRR**.
  - **Se miran juntas**, con el contexto al lado. Ninguna métrica sola cuenta la historia completa.

#### ⚠️ Lo que el MRR **no** hace

- Cuenta **solo el primer acierto**: no le importa si hay otros correctos después.
- **Depende de `k`**: con `k = 3`, una respuesta que quedó 4.ª vale 0.
- **No es un porcentaje de aciertos.** 0.625 **no** significa "62.5 % de aciertos" (eso sería `hit@k`). Es un promedio de `1 ÷ puesto`.
- No explica **por qué** falló una pregunta: para eso mira la lista de resultados (la tabla de más abajo) o el diagnóstico por capas de la Sesión 7.

#### ✏️ Tu turno (3 minutos)

1. ¿Cuál es el MRR si el primer acierto quedó en los puestos **1, 2, 3 y 4**?
2. En el ejemplo `[1, 1, 2, None]`, la pregunta 4 mejora: pasa de *"no apareció"* al **puesto 3**. ¿Cuánto sube el MRR?
3. Con `k = 3` y un RAG que envía los 3 fragmentos al LLM, ¿preferirías el buscador **B** o el **C**? Defiéndelo en una frase.

<details><summary>Ver soluciones</summary>

1. `(1 + 0.5 + 0.333 + 0.25) ÷ 4` = **0.521**.
2. Pasa de `2.5 ÷ 4 = 0.625` a `(1 + 1 + 0.5 + 0.333) ÷ 4` = **0.708**: sube **0.083** (un tercio de acierto repartido entre 4 preguntas).
3. Es una pregunta de criterio. **B** garantiza que la respuesta llega al prompt (`hit@3` = 100 %), pero siempre en el último lugar y el modelo puede darle menos peso; **C** acierta arriba, pero falla por completo una de cuatro. Con `k = 3`, la mayoría preferirá **B** *si el LLM lee bien el contexto completo* y **C** *si el ruido de los primeros fragmentos lo confunde*. Lo que no se puede es decidir con una sola métrica: se mide con ambas y con respuestas reales.

</details>

Prueba tus propias posiciones (cámbialas y ejecuta):

In [ ]:
mis_posiciones = [1, 2, 3, 4]      # ← cámbialas: un número por pregunta, o None si la respuesta no apareció
_ = mrr_paso_a_paso(mis_posiciones)

Esta es nuestra **línea base**: la recuperación de la Sesión 5 (fragmentos de ≈70 palabras por oraciones, búsqueda semántica) sobre el corpus nuevo.

In [ ]:
def linea(nombre, r):
    """Imprime una fila de resultados."""
    print(f"{nombre:<38} hit@1 {r['hit1']:>2}/{r['n']:<2} · hit@{r['k']} {r['hitk']:>2}/{r['n']:<2} · MRR {r['mrr']:.2f} · contexto ≈ {r['contexto']:.0f} palabras")

if HAY_EMB:
    fragmentos_base = fragmentar_corpus(documentos, estrategia="oraciones", max_palabras=70)
    indice_base = construir_indice(fragmentos_base)        # vectoriza (local, gratis) y arma el índice
    print("Fragmentos:", resumen_fragmentos(fragmentos_base))
    linea("Línea base (oraciones 70, semántica)",
          evaluar_busqueda(BASE, lambda p, k: buscar_vectorial(indice_base, p["pregunta"], k), k=3))

Ese número es **tu punto de partida**. Cada técnica de las partes siguientes tiene que **superarlo** (o justificar su costo) para ganarse un lugar en el pipeline.

**Qué mirar:** las filas con `—` (la respuesta no apareció entre los 3 primeros: aportan 0) y las de puesto 2 o 3 (aportan 0.50 o 0.33). **Ahí** está lo que le falta al MRR para llegar a 1.0, y esas son las preguntas que las técnicas de las partes siguientes tienen que rescatar.

**Con `nomic-embed-text-v2-moe`** (el modelo de la clase; con otro modelo los puestos pueden cambiar): **18 preguntas** quedan primeras (valen 1.00), **2** quedan terceras (0.33 cada una), **1** queda segunda (0.50) y **1** no aparece (0). La suma es `18 + 0.33 + 0.33 + 0.50 + 0 = 19.17`, y `19.17 ÷ 22 = 0.87`.

Fíjate en lo que cuenta cada cosa: `hit@1 = 18/22` (solo las 18 primeras), `hit@3 = 21/22` (las 18 + las 3 que quedaron 2.ª o 3.ª) y `MRR = 0.87` (además **pondera** cuán arriba quedó cada una). Las tres métricas son consistentes entre sí, pero **cada una cuenta una parte distinta de la historia**.

In [ ]:
if HAY_EMB:
    filas = []
    for p in BASE:
        res = buscar_vectorial(indice_base, p["pregunta"], 3)
        puesto = next((i for i, r in enumerate(res, start=1) if contiene_clave(r.texto, p["clave"])), None)
        filas.append((p["id"], p["tipo"], puesto, 1 / puesto if puesto else 0.0, p["pregunta"]))

    print(f"{'id':<5}{'tipo':<11}{'puesto':>8}{'1 ÷ puesto':>12}   pregunta")
    for id_, tipo, puesto, rr, pregunta in filas:
        print(f"{id_:<5}{tipo:<11}{(puesto if puesto else '—')!s:>8}{rr:>12.2f}   {pregunta[:58]}")
    suma = sum(f[3] for f in filas)
    print(f"\nMRR = {suma:.2f} ÷ {len(filas)} preguntas = {suma / len(filas):.2f}")
else:
    print("⚪ Esta celda necesita el servidor de embeddings (ver la Parte 0).")

## 🗺️ Mapa de la sesión: el pipeline y qué problema ataca cada pieza

Antes de entrar al detalle, mira el recorrido completo. Cada caja existe porque **un problema real** la exige; si no tienes ese problema, no necesitas esa caja.

```
INGESTA (se hace una vez, o cuando el documento cambia)
 archivo ─► [2 · EXTRAER texto + estructura] ─► [2 · OCR si es escaneo] ─► [3-4 · FRAGMENTAR] ─► vectorizar ─► guardar + metadatos
              problema: formatos distintos,       problema: la página es      problema: cortes que       (+ índice BM25)
              basura (menús, pies de página)      una imagen, no texto        cortan o diluyen ideas

CONSULTA (se hace en cada pregunta)
 pregunta ─► [9 · REESCRIBIR]  ─► [5-6 · BUSCAR: semántica + BM25 + RRF] ─► [7 · FILTRO] ─► [8 · RERANK] ─► prompt con citas ─► LLM
   problema: "¿y los domingos?"     problema: códigos exactos Y paráfrasis     problema: quién      problema: el       (Sesión 5)
   no se entiende sola              (cada método tiene un punto ciego)          pregunta cambia      correcto va 5.º
                                                                                la respuesta         y enviamos 3

                                [1 · MEDIR: hit@k, MRR, contexto]  ◄── se aplica a TODAS las cajas
```

| Pregunta que te haces | Pieza | Parte |
| --- | --- | --- |
| ¿Cómo sé si algo mejoró? | Métricas sobre un conjunto de preguntas | 1 |
| ¿El texto salió limpio de mi PDF/Word/web? | Ingesta (y OCR); parser con *layout*: Docling | 2 |
| ¿Mis fragmentos son legibles y del tamaño adecuado? | Chunking | 3 y 4 |
| ¿Encuentra un código de error exacto? | BM25 | 5 |
| ¿Encuentra tanto códigos como paráfrasis? | Híbrida + RRF | 6 |
| ¿Ve solo lo que ese usuario puede ver? | Metadatos y filtros | 7 |
| ¿El mejor fragmento llega entre los que envío? | Reranking | 8 |
| ¿Entiende preguntas de seguimiento o vagas? | Reescritura, multi-consulta, HyDE | 9 |

> **Regla de la sesión:** cada caja se **mide** antes de quedarse. Las cajas con LLM (rerank, reescritura, multi-consulta) añaden latencia, dinero y un punto de falla.

---
# Parte 2 · Ingesta multiformato

## El paso más subestimado

Antes de fragmentar y vectorizar hay que **extraer el texto**. Si en esta etapa entra basura (menús de navegación, pies de página, tablas aplanadas, texto de otra columna), **todo lo demás trabaja con basura**: *garbage in, garbage out*.

```
archivo ─► extraer texto + estructura ─► Documento(secciones) ─► fragmentar ─► vectorizar ─► guardar
```

`ingesta.py` convierte cada formato en la **misma estructura** que ya conoces (`Documento` con secciones `(título, texto)`) y añade `tipo` y `avisos`:

| Formato | Herramienta | Qué hay que cuidar |
| --- | --- | --- |
| **Markdown** | El lector de la Sesión 5 | Casi nada: ya trae estructura |
| **HTML** | `BeautifulSoup` | Quitar `<script>`, `<style>`, menús y pie de página; conservar títulos y listas |
| **DOCX** | `python-docx` | Respetar los estilos de título (*Heading 1/2*); las **tablas** hay que convertirlas |
| **PDF** (con texto) | `pypdf` | Una sección por **página**; las líneas se cortan por el ancho de la hoja; las **tablas se aplanan** |
| **PDF escaneado** | **OCR** (con Docling, en 2.5) | No hay texto: es una imagen |

> 🎧 **Audio y video también son fuentes.** Una charla grabada o un video de capacitación contienen conocimiento que ningún lector de arriba ve. Se **transcriben con Whisper** (un modelo de voz a texto) y esa transcripción entra al mismo pipeline: se fragmenta, se vectoriza y se busca. Lo veremos en la siguiente sesión.

Al final de esta Parte (2.5) verás **Docling**, un parser moderno que lee **todos** estos formatos con un solo convertidor, reconstruye las **tablas** y trae su propio **OCR**.

### Ingesta y parsing, aterrizados

**🧠 Qué es.** La **ingesta** es todo el camino que convierte archivos en fragmentos buscables: *extraer → fragmentar → vectorizar → guardar*. El **parsing** es el primer tramo: sacar el **texto** y su **estructura** (títulos, secciones, páginas, tablas) de un formato de archivo.

**⚙️ Qué hace.** Cada formato guarda la información de forma distinta, así que hay un **lector por formato** que entrega siempre lo mismo: un `Documento` con una lista de `(título, texto)` y avisos.

**🎯 Para qué sirve.** Es la **puerta de entrada**: lo que no entra limpio por aquí **no se puede arreglar después**. Un menú de navegación que se vectoriza compite con el contenido real; una tabla aplanada pierde sus filas; una página escaneada no entra.

**🏗️ Analogía de infraestructura.** Es el **parser del pipeline de logs** (el *grok* o el *shipper* que da formato a los logs antes de indexarlos en Elasticsearch). Si el parser mete líneas de un stack trace como eventos sueltos, o descarta un campo, ningún dashboard ni alerta podrá compensarlo. *Garbage in, garbage out.*

**✏️ Ejemplo a mano.** Una página web tiene 3 cosas: el **contenido** ("El plan Familiar cuesta 39 dólares"), un **menú** ("Inicio · Planes · Contacto") y un **pie** ("Síguenos en redes sociales"). Si el lector no separa, los tres se vectorizan. Una pregunta sobre "redes" o "contacto" puede traer el pie y el menú como si fueran respuesta. En el ejemplo de esta sesión, la guía técnica pasa de 956 a 839 palabras al descartar el ruido.

| Si el lector falla en… | Consecuencia para el RAG |
| --- | --- |
| Descartar menús/scripts (HTML) | Basura vectorizada; respuestas que citan el pie de página |
| Respetar títulos (DOCX) | Se pierde la sección → citas vagas y metadatos mal etiquetados |
| Tablas (DOCX/PDF) | La fila queda separada de su encabezado o aplanada |
| Páginas sin texto (PDF escaneado) | El documento **no existe** para el RAG, y nadie se entera |

**⚠️ Cuándo no complicarse.** Si tus documentos ya son Markdown limpio o texto plano, el lector es trivial. Subir de nivel (extractores de tablas, *document AI*, modelos con visión, parsers con análisis de *layout* como Docling, en 2.5) solo se justifica cuando **mides** que las tablas o el diseño te están costando respuestas.

**🐞 Errores típicos.** (1) Asumir que "se leyó bien" sin mirar una muestra del texto extraído. (2) No tener **avisos** (páginas vacías, secciones de 0 palabras). (3) Perder la estructura de títulos y luego no poder citar ni filtrar. (4) Ingerir una vez y olvidar que, si el documento cambia, hay que **reingestar**.

> **Regla práctica:** después de cada lector nuevo, imprime **una muestra del texto extraído** y léela con tus ojos. Es la prueba más barata y la que más errores atrapa.

## 2.1 HTML: quedarse con el contenido, descartar el ruido

In [ ]:
crudo = Path("datos/guia_tecnica.html").read_text(encoding="utf-8")
print("HTML crudo:", len(crudo.split()), "palabras, incluidas las etiquetas")
print("Contiene <script>:", "<script>" in crudo, "· <nav>:", "<nav>" in crudo, "· <footer>:", "<footer>" in crudo)

html = leer_documento_multi("datos/guia_tecnica.html")
todo = " ".join(t for _, t in html.secciones)
print("\nDespués de extraer:", len(todo.split()), "palabras")
print("¿Quedó 'Síguenos en redes sociales' (pie de página)?", "Síguenos" in todo)
print("¿Quedó 'gtag' (script)?", "gtag" in todo)
print("\nSecciones:", [t for t, _ in html.secciones])
print("\nUna sección con lista (se conserva la lista, con guiones):\n")
print(dict(html.secciones)["Códigos de error del router"][:420], "…")

## 2.2 DOCX: estilos de título y tablas

Un Word bien hecho usa **estilos** (*Título 1*, *Título 2*): son la estructura que necesitamos. Nuestro lector arma el nombre de la sección con ambos niveles (`Parte A · Clientes residenciales › Compensaciones`). Ese nombre **no es decoración**: de él sale el metadato `publico` que usaremos para **filtrar** (Parte 7).

Las **tablas** no son texto corrido: las convertimos en una línea por fila (`celda | celda | celda`).

In [ ]:
docx = leer_documento_multi("datos/condiciones_servicio.docx")
for titulo, texto in docx.secciones:
    print(f"• {titulo}  ({len(texto.split())} palabras)")
print("\nLa sección con tabla:\n")
print(dict(docx.secciones)["Parte A · Clientes residenciales › Penalización por cancelación anticipada"])

## 2.3 PDF con texto: cuidado con las tablas

`pypdf` entrega el texto en el orden en que lo guardó el PDF. Un PDF **no tiene filas ni columnas**: tiene letras en posiciones. Por eso la tabla de fechas del boletín **se aplana**: la información sigue ahí, pero ya no es una tabla.

In [ ]:
pdf = leer_documento_multi("datos/boletin_mantenimiento.pdf")
print([t for t, _ in pdf.secciones])
pagina1 = pdf.secciones[0][1]
inicio = pagina1.index("Ventanas por zona")
print("\nLa tabla de ventanas, tal como quedó:\n")
print(pagina1[inicio:inicio + 260], "…")

Así, **la fila de la zona sur queda en el mismo renglón que las demás**. Si el fragmento se corta en mala hora, la fecha puede separarse de su zona. Para PDFs con muchas tablas se usan extractores que reconocen la estructura (por ejemplo `pdfplumber`, o servicios de *document AI*), o se pide a un modelo con visión que lea la página. **En 2.5 verás cómo Docling reconstruye esta tabla.**

## 2.4 PDF escaneado: cuando no hay texto

`acta_escaneada.pdf` es **una imagen de un papel**. No tiene capa de texto: `pypdf` no devuelve nada, y nuestro lector lo **avisa** en lugar de ingerir un documento vacío en silencio.

In [ ]:
acta = leer_documento_multi("datos/acta_escaneada.pdf")
print("Secciones extraídas:", len(acta.secciones))
print("Avisos:", acta.avisos)

### ¿Qué es el OCR?

**OCR** (*Optical Character Recognition*, reconocimiento óptico de caracteres) es convertir una **imagen de texto** en **texto**. Es el paso que falta antes de ingerir escaneos, fotos de documentos o PDFs de "imprimir a imagen".

| Opción | Cuándo | Ojo con |
| --- | --- | --- |
| **Tesseract** (código abierto, local) | Texto impreso limpio; privacidad total | Rinde mal con fotos torcidas, manuscritos y tablas |
| **OCR de la nube** (Azure Document Intelligence, Google Document AI, AWS Textract) | Volumen, tablas, formularios | 💰 pago por página y los documentos salen de tu red |
| **Modelo con visión** (un LLM que "ve" la página) | Documentos complejos, mezcla de tabla y texto | 💰 más caro por página y puede equivocarse con cifras: **verifica** |

> **Aquí solo lo detectamos** (lo que hace `ingesta.py`). En la sección **2.5** lo ejecutamos de verdad con Docling, que trae su propio motor de OCR. Lo importante es **detectarlo** y **decidir** qué hacer: un escaneo ingerido sin OCR **no existe** para tu RAG, y nadie te avisará de que le falta información.

**Reto:** ¿qué pasaría con tu RAG si el 20 % de tus PDFs fueran escaneos y nadie lo notara? Escribe cómo lo detectarías (pista: cuenta las páginas sin texto en cada ingesta).

#### OCR, aterrizado

**🧠 Qué es.** Un motor que **mira una imagen** (un escaneo, una foto, un PDF "impreso a imagen") y **devuelve el texto** que cree leer, normalmente con un **nivel de confianza** por palabra.

**⚙️ Qué hace.** Recorre tres pasos: *preprocesar* la imagen (enderezar, limpiar, binarizar), *reconocer* las letras y *armar* líneas y párrafos. Su resultado es texto **con posibles errores**, no texto perfecto.

**🎯 Para qué sirve.** Para que un papel escaneado deje de ser una foto y pase a ser un documento más del RAG.

**🏗️ Analogía de infraestructura.** Es **digitalizar un runbook que solo existe impreso**: antes de poder buscar en él, alguien (o algo) tiene que transcribirlo. Y como toda transcripción, hay que **revisar las cifras y los códigos**.

**✏️ Ejemplo a mano: por qué importa la exactitud.** Si el OCR lee `E-217` como `E-2l7` (una "l" minúscula en vez de un 1), la búsqueda por código exacto (**BM25**, Parte 5) **no lo encuentra**, y la semántica tampoco. El texto "existe" pero ya no coincide. Con una cifra pasa lo mismo: `39,00` leído como `3900` cambia la respuesta.

| Opción | Úsala cuando | Ojo con |
| --- | --- | --- |
| Local (Tesseract) | Texto impreso limpio y privacidad total | Fotos torcidas, manuscritos, tablas |
| Servicio en la nube | Volumen, formularios y tablas | Costo por página; el documento sale de tu red |
| Modelo con visión | Documento complejo (tabla + texto + sellos) | Más caro, y puede "corregir" o inventar una cifra |

**⚠️ Cuándo no.** Si el PDF **ya tiene capa de texto**, no hace falta OCR: usarlo solo añade errores y costo. Se detecta porque el extractor devuelve texto; si devuelve vacío en una página, ahí sí.

**🐞 Errores típicos.** (1) No detectar los escaneos y descubrir meses después que "faltaban" documentos. (2) Confiar ciegamente en las cifras del OCR. (3) Tirar el original: **conserva el archivo** como fuente de la cita, para que una persona pueda verificar.

## 2.5 Docling: layout, tablas y OCR

Hasta aquí hay **un lector por formato** (2.1 a 2.4), cada uno con su trampa, y dos problemas siguen sin resolverse bien: la **tabla del PDF se aplanó** (2.3) y el **escaneo no tiene texto** (2.4). Esta sección presenta un **parser moderno** que ataca los dos con una sola herramienta: **Docling**.

> 📦 **Es opcional y pesado.** Docling (y PyTorch, solo CPU) **no** forman parte del entorno base: se instalan con `uv sync --group docling` y, la primera vez, descargan **varios cientos de MB de modelos**. Hazlo **antes de clase**. Si no está instalado, las celdas de esta sección (y las de 3.4 y 4.2) muestran un aviso con el comando exacto y **el resto del notebook funciona igual**.

### Por qué un PDF "no tiene párrafos ni tablas"

Un PDF es una lista de instrucciones de dibujo: *"escribe la letra A en la posición (40, 100)"*. No existe un objeto "párrafo", "fila" o "columna": solo hay **letras en posiciones**. Un extractor simple (`pypdf`) las lee **en el orden en que se guardaron** y las pega. De ahí salen tres síntomas: líneas cortadas por el ancho de la hoja, **tablas aplanadas** en una sola línea, y páginas a dos columnas con el texto mezclado (**orden de lectura** equivocado).

### El análisis de *layout*

El **análisis de layout** (*document layout analysis*) hace lo que hace tu ojo: mira la página **como una imagen**, dibuja **cajas** alrededor de cada región y les pone un **tipo** (título, párrafo, lista, tabla, figura, pie de página). Después **ordena** las regiones en un orden de lectura y, dentro de las tablas, otro modelo reconoce filas, columnas y celdas combinadas. Docling hace ese recorrido con modelos de visión que corren en CPU:

```
página (PDF o imagen)
   ├─ 1. LAYOUT      modelo de visión: ¿qué regiones hay y de qué tipo?
   ├─ 2. ORDEN       orden de lectura de las regiones (columnas, notas…)
   ├─ 3. TABLAS      modelo de tablas (TableFormer): filas, columnas y celdas combinadas
   ├─ 4. OCR         SOLO donde la región es una imagen sin texto
   └─ 5. RESULTADO   un DoclingDocument: jerarquía de títulos, textos, tablas y la página de cada elemento
                     → se exporta a Markdown o JSON y queda listo para fragmentar
```

Con Markdown, HTML y DOCX no hace falta mirar ninguna página: el formato **ya trae** su estructura y Docling la lee directamente (por eso son casi instantáneos). El análisis de layout y el OCR solo se pagan en **PDF e imágenes**.

### Tres tipos de PDF (y qué hacer con cada uno)

| Tipo de PDF | Cómo se reconoce | Qué usar |
| --- | --- | --- |
| **Con texto** (generado por un programa) | El extractor devuelve texto | Extractor directo, o Docling **sin OCR** si hay tablas o columnas |
| **Escaneado** (una imagen por página) | Cero texto en la página | **OCR** |
| **Mixto** (unas páginas con texto, otras escaneadas, o texto sobre una imagen) | Hay páginas con texto y páginas vacías | Decidir **por página**: OCR solo donde falta el texto |

### Motores de OCR

| Motor | Dónde corre | Comentario |
| --- | --- | --- |
| **RapidOCR** | Local | El que Docling usa por defecto: viene incluido, **no hay que instalar nada más** |
| **EasyOCR**, **Tesseract** | Local | Alternativas que Docling puede usar como extra (Tesseract necesita el programa instalado en el sistema) |
| **OCR de la nube** | Servicio | Mejor con volumen, formularios y tablas; 💰 por página y el documento sale de tu red |
| **Modelo con visión** | Servicio o local | Para documentos complejos; más caro y puede "corregir" una cifra: verifica |

**¿Cuándo hace falta OCR?** Cuando la página **no tiene capa de texto**. No se aplica "por si acaso": añade tiempo y **errores** que un PDF con texto no tiene. La estrategia práctica es **primero sin OCR y reintentar con OCR solo si casi no hay texto**.

**¿Cuánto cuesta?** Dinero, nada (todo es local). **Tiempo**, sí: la primera página carga los modelos (segundos) y cada página de escaneo añade el costo del OCR. Y **espacio**: los modelos pesan cientos de MB.

### Docling y el análisis de layout, aterrizados

**🧠 Qué es.** Un convertidor de documentos que **entiende el diseño de la página**: reconoce títulos, párrafos, listas, tablas y figuras, reconstruye el orden de lectura y, si hace falta, aplica OCR. Una sola interfaz para Markdown, HTML, DOCX, PDF e imágenes.

**⚙️ Qué hace.** Convierte el archivo en un **`DoclingDocument`**: un árbol con la jerarquía de títulos, cada texto con su tipo, cada tabla con sus filas y columnas, y la **página** de cada elemento. De ahí se exporta a Markdown o se fragmenta (3.4).

**🎯 Para qué sirve.** Para no escribir ni mantener un lector por formato, y sobre todo para recuperar lo que un extractor de texto **pierde**: tablas con su estructura, orden de lectura correcto y texto de escaneos.

**🏗️ Analogía de infraestructura.** Pasar de hacer `grep` sobre logs en texto plano a usar un **parser con esquema** (el que conoce el formato y devuelve campos). El primero ve líneas; el segundo entiende qué es cada campo y cuál es su tipo. Y el OCR es como **capturar tráfico de una interfaz que no exporta logs**: hay que reconstruir los datos desde lo que se ve, con margen de error.

**✏️ Ejemplo a mano.** En la celda de abajo, una "página" de juguete es una lista de palabras con sus coordenadas, guardadas **columna por columna**. Leídas en ese orden salen mezcladas (*Zona Norte Sur Dia 5 12…*). Agrupando por altura (filas) y ordenando por posición horizontal (columnas) reaparece la tabla. Eso, con un modelo de visión que además ve bordes, celdas combinadas y varias columnas, es el análisis de layout.

**⚠️ Cuándo no.** Con Markdown limpio, HTML sencillo o PDFs de texto corrido **sin tablas ni columnas**, los lectores de 2.1 a 2.3 son más livianos, más rápidos y no instalan nada. Docling cuesta tiempo de CPU, cientos de MB de modelos y una dependencia grande: se justifica cuando **mides** que las tablas, el orden de lectura o los escaneos te cuestan respuestas.

**🐞 Errores típicos.** (1) Activar OCR en PDFs que ya tienen texto: añade errores sin necesidad. (2) Fiarse del OCR sin revisar códigos y cifras (lo vemos en 2.5.3). (3) No instalar los modelos **antes** de clase y descubrir la descarga en vivo. (4) Dar por buena una tabla sin mirar cómo quedó. (5) Olvidar guardar si un texto salió de OCR: conviene saberlo para revisarlo con más cuidado.

**🧾 De dónde viene.** Se creó en **IBM Research Zurich**, se abrió en 2024 y hoy lo aloja la **LF AI & Data Foundation**. El código es **MIT**; los **modelos** que descarga tienen cada uno **su licencia** (revísalas antes de llevarlo a un producto).

**🚫 Qué NO hace.** No vectoriza ni busca: solo **prepara** el documento. No garantiza un OCR perfecto (lo vamos a medir en 2.5). En la versión probada, su transcripción de audio **no conserva los tiempos** en el documento. Y no es liviano: varios cientos de MB de modelos la primera vez. Con Markdown, HTML o PDF de texto corrido, un lector simple alcanza.

In [ ]:
# Ejemplo a mano: un PDF no guarda filas ni columnas, guarda PALABRAS CON COORDENADAS (sin Docling, sin costo)
# Datos de juguete: (texto, x, y). Es el ORDEN EN QUE EL PROGRAMA QUE CREÓ EL PDF DIBUJÓ cada palabra:
# columna por columna (algo muy común). Lo que ve el ojo en la página es otra cosa.
palabras = [
    ("Zona", 40, 100), ("Norte", 40, 120), ("Sur", 40, 140),
    ("Dia", 160, 100), ("5", 160, 120), ("12", 160, 140),
    ("Horario", 240, 100), ("01:00", 240, 120), ("00:00", 240, 140),
]

print("1) Leer en el orden en que está guardado (lo que hace un extractor simple):")
print("  ", " ".join(t for t, x, y in palabras))

# 2) El "layout" en miniatura: agrupar por altura (y) = las FILAS, y ordenar cada fila por x = las COLUMNAS.
filas = {}
for t, x, y in palabras:
    filas.setdefault(round(y / 20), []).append((x, t))        # casi la misma altura → la misma fila
tabla = [[t for x, t in sorted(fila)] for _, fila in sorted(filas.items())]

print("\n2) Reconstruir filas y columnas con las coordenadas:")
for fila in tabla:
    print("   | " + " | ".join(f"{celda:<8}" for celda in fila) + " |")

In [ ]:
from parsing_docling import COMANDO_INSTALAR, docling_disponible

HAY_DOCLING, msg_docling = docling_disponible()
print(("✅ " if HAY_DOCLING else "⚠️ ") + msg_docling)
if not HAY_DOCLING:
    print(f"\nPara activar las celdas de Docling (2.5, 3.4 y 4.2), ejecuta en una terminal, dentro de esta carpeta:\n"
          f"    {COMANDO_INSTALAR}\nY reinicia el kernel. Mientras tanto esas celdas se omiten y el resto funciona igual.")

### 2.5.1 Una tabla en un PDF con texto: pypdf contra Docling

Mismo archivo que en 2.3 (`boletin_mantenimiento.pdf`), dos lectores. Como el PDF **sí tiene texto**, Docling no necesita OCR: lo desactivamos (`ocr=False`). Fíjate en tres cosas: cómo queda la tabla, qué **tipo** le asigna Docling a cada región y en qué **página** está cada una.

In [ ]:
if not HAY_DOCLING:
    print(f"⚠️ Celda omitida: falta Docling. Instálalo con `{COMANDO_INSTALAR}` y reinicia el kernel (el resto del notebook sigue funcionando).")
else:
    from parsing_docling import leer_con_docling

    t0 = time.perf_counter()
    boletin = leer_con_docling("datos/boletin_mantenimiento.pdf", ocr=False)   # tiene capa de texto: no hace falta OCR
    seg = time.perf_counter() - t0
    print(f"Docling leyó el boletín en {seg:.0f} s (la primera vez incluye cargar los modelos): "
          f"{len(boletin.pages)} páginas · {len(boletin.texts)} bloques de texto · {len(boletin.tables)} tabla")

    print("\n━━ pypdf (el lector de 2.3): la tabla, aplanada en una línea ━━")
    pagina1 = leer_documento_multi("datos/boletin_mantenimiento.pdf").secciones[0][1]
    inicio = pagina1.index("Ventanas por zona")
    print(pagina1[inicio:inicio + 190], "…")

    print("\n━━ Docling: la misma tabla, reconstruida (exportada a Markdown) ━━")
    tabla_dl = boletin.tables[0]
    print(tabla_dl.export_to_markdown(doc=boletin))
    print(f"\n→ {tabla_dl.data.num_rows} filas × {tabla_dl.data.num_cols} columnas, en la página {tabla_dl.prov[0].page_no}")

    print("\n━━ Lo que Docling reconoce en las páginas: tipo de región, en orden de lectura ━━")
    for item, _ in boletin.iterate_items():
        etiqueta = str(item.label).split(".")[-1]
        texto = (getattr(item, "text", "") or "[tabla]")[:52].replace("\n", " ")
        print(f"   p.{item.prov[0].page_no} · {etiqueta:<15} {texto}")

**Qué ves.** `pypdf` entrega la tabla como **una sola línea**: la información está, pero ya no hay filas ni encabezados. Docling la devuelve como una tabla de **4 filas × 3 columnas** (el encabezado más las tres zonas), con cada fecha **al lado de su zona**, y además lista las regiones con su tipo (`section_header`, `text`, `table`) **en orden de lectura** y con su página. Eso sirve para dos cosas: que un fragmento no separe la fecha de su zona, y poder **citar con página y sección**.

**El precio.** Docling tarda segundos la primera vez en este PDF (carga los modelos de layout y de tablas; en esta máquina fueron 7 s) frente a milisegundos de `pypdf`. Para un PDF sin tablas ni columnas, ese costo no se justifica.

### 2.5.2 El acta escaneada: OCR con Docling

En 2.4 vimos que `pypdf` no devuelve nada y que nuestro lector lo **avisa**. Docling incluye un motor de OCR (**RapidOCR**, local) que viene activado por defecto. Probamos tres cosas: Docling **sin** OCR, Docling **con** OCR y la estrategia "primero sin OCR, y OCR solo si hace falta".

In [ ]:
if not HAY_DOCLING:
    print(f"⚠️ Celda omitida: falta Docling. Instálalo con `{COMANDO_INSTALAR}` y reinicia el kernel (el resto del notebook sigue funcionando).")
else:
    from parsing_docling import leer_con_docling, leer_pdf_adaptativo

    # a) Sin OCR: Docling tampoco puede leer una imagen como si fuera texto.
    sin_ocr = leer_con_docling("datos/acta_escaneada.pdf", ocr=False)
    print(f"Sin OCR: {len(sin_ocr.texts)} bloques de texto (igual que pypdf: la página es una imagen).")

    # b) Con OCR (el predeterminado): la primera vez carga los modelos de OCR, por eso tarda más.
    t0 = time.perf_counter()
    con_ocr = leer_con_docling("datos/acta_escaneada.pdf", ocr=True)
    seg = time.perf_counter() - t0
    TEXTO_ACTA = " ".join(con_ocr.export_to_markdown().split())
    print(f"Con OCR: {len(TEXTO_ACTA.split())} palabras en {seg:.0f} s (incluye cargar los modelos de OCR).\n")
    print("Texto reconocido:\n  ", TEXTO_ACTA)
    terminos = tokenizar(TEXTO_ACTA)
    print("\n¿BM25 tiene el código 'ax-2200' como término?", "ax-2200" in terminos, "· términos parecidos:", [t for t in terminos if "ax" in t])

    # c) La estrategia práctica: primero sin OCR (rápido) y, si casi no hay texto, reintentar con OCR.
    print("\nEstrategia 'primero sin OCR, y con OCR solo si hace falta':")
    for nombre in ("boletin_mantenimiento.pdf", "acta_escaneada.pdf"):
        t0 = time.perf_counter()
        _, uso_ocr = leer_pdf_adaptativo(f"datos/{nombre}")
        print(f"   {nombre:<28} → {'necesitó OCR' if uso_ocr else 'texto directo, sin OCR'} ({time.perf_counter() - t0:.1f} s, con los modelos ya cargados)")

**Qué ves.**
- **Sin OCR:** 0 bloques de texto, igual que `pypdf`. Docling tampoco convierte una imagen en letras por arte de magia.
- **Con OCR:** el acta completa, 21 palabras, que empiezan por *"ACTA DE INSTALACION (COPIA ESCANEADA) Cliente: Familia Rojas…"*. Tardó unos segundos porque cargó los modelos de OCR (3 s en esta máquina después de haber leído ya el boletín; en una corrida donde fue lo primero que se ejecutó tardó entre 15 y 21 s).
- **La estrategia práctica** distingue los dos PDF: el boletín se lee directo y el acta necesita OCR.
- **Un detalle que importa:** aun con este texto "perfecto", el OCR pegó `AX-2200` con la palabra siguiente (`AX-2200Metros`). Para BM25 el término ya no es `ax-2200` sino `ax-2200metros`, así que **una búsqueda por el código del equipo no lo encontraría**. Es un error silencioso: el texto "se ve bien".

### 2.5.3 Lección: el OCR se equivoca (y por qué importa)

El acta de `datos/` es un escaneo **limpio y sintético**, casi el mejor caso posible. Los escaneos reales vienen **torcidos, con ruido y a baja resolución**. Para verlo, la celda siguiente **genera con Pillow** versiones degradadas del acta (inclinación, desenfoque, ruido, compresión) y le pasa a cada una el mismo OCR. Cada nivel se compara con el texto reconocido en el acta original, que usamos como "texto bueno". Las imágenes de prueba se guardan en una carpeta temporal, **no** en `datos/`.

| Nivel | Qué le hacemos a la imagen |
| --- | --- |
| **limpia (control)** | Nada: la misma página, recortada. Comprueba que el montaje no introduce errores |
| **leve** | 2° de inclinación, la mitad de píxeles, algo de ruido |
| **media** | 3°, 35 % de los píxeles, más ruido y desenfoque |
| **fuerte** | 4°, 25 % de los píxeles, mucho ruido |

In [ ]:
if not HAY_DOCLING:
    print(f"⚠️ Celda omitida: falta Docling. Instálalo con `{COMANDO_INSTALAR}` y reinicia el kernel (el resto del notebook sigue funcionando).")
else:
    import difflib
    import tempfile

    import numpy as np
    import pypdfium2 as pdfium
    from IPython.display import display
    from PIL import Image, ImageChops, ImageFilter

    CARPETA_TMP = Path(tempfile.mkdtemp(prefix="acta_ocr_"))      # las imágenes de prueba NO se guardan en datos/

    def pagina_como_imagen(ruta_pdf, escala=3):
        """Renderiza la página 1 del PDF como imagen (gris) y recorta los márgenes blancos."""
        img = pdfium.PdfDocument(str(ruta_pdf))[0].render(scale=escala).to_pil().convert("L")
        x0, y0, x1, y1 = ImageChops.invert(img).getbbox()
        return img.crop((x0 - 40, y0 - 40, x1 + 40, y1 + 40))

    def degradar(img, grados, escala, ruido, desenfoque, calidad_jpeg, semilla=7):
        """Simula un mal escaneo: inclinación, baja resolución, desenfoque, ruido y compresión JPEG fuerte."""
        import io
        a = img.rotate(grados, expand=True, fillcolor=255, resample=Image.BICUBIC)
        a = a.resize((int(a.width * escala), int(a.height * escala)), Image.BILINEAR)
        a = a.filter(ImageFilter.GaussianBlur(desenfoque))
        gris = np.asarray(a, dtype=np.float32) + np.random.default_rng(semilla).normal(0, ruido, (a.height, a.width))
        buffer = io.BytesIO()
        Image.fromarray(np.clip(gris, 0, 255).astype("uint8")).save(buffer, "JPEG", quality=calidad_jpeg)
        return Image.open(buffer).convert("L")

    def limpiar_ocr(texto):
        return " ".join(texto.replace("## ", "").split())

    def ocr_de_imagen(img, nombre):
        """Guarda la imagen como un PDF de una sola imagen (un 'escaneo') y le aplica el OCR de Docling."""
        ruta = CARPETA_TMP / f"{nombre}.pdf"
        img.convert("RGB").save(ruta, resolution=150)
        t0 = time.perf_counter()
        texto = limpiar_ocr(leer_con_docling(ruta).export_to_markdown())
        return texto, time.perf_counter() - t0

    def diferencias(bueno, ocr):
        """Las palabras que cambiaron, como 'Rojas → Fojas'."""
        if not ocr:
            return ["(no devolvió texto)"]
        a, b = bueno.split(), ocr.split()
        return [f"{' '.join(a[i1:i2]) or '∅'} → {' '.join(b[j1:j2]) or '∅'}"
                for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b).get_opcodes() if op != "equal"]

    BUENO = limpiar_ocr(leer_con_docling("datos/acta_escaneada.pdf").export_to_markdown())   # el texto de referencia
    limpia = pagina_como_imagen("datos/acta_escaneada.pdf")
    # (grados, escala, ruido, desenfoque, calidad_jpeg)
    NIVELES = {"limpia (control)": None, "leve": (2, 0.50, 18, 0.6, 50), "media": (3, 0.35, 25, 0.8, 35),
               "fuerte": (4, 0.25, 35, 1.0, 25)}
    IMAGENES, RESULTADOS_OCR = {}, {}
    print(f"{'nivel':<18}{'píxeles':>10}{'seg':>6}{'parecido':>10}   palabras que cambiaron")
    for nivel, params in NIVELES.items():
        img = limpia if params is None else degradar(limpia, *params)
        texto, seg = ocr_de_imagen(img, nivel.split()[0])
        IMAGENES[nivel], RESULTADOS_OCR[nivel] = img, texto
        parecido = difflib.SequenceMatcher(None, BUENO, texto).ratio()
        cambios = diferencias(BUENO, texto)
        print(f"{nivel:<18}{img.width:>5}×{img.height:<4}{seg:>5.1f}{parecido:>9.0%}   {'; '.join(cambios) or '—'}")

    # Lo que ve el OCR: la misma acta, a tamaño real de píxeles, de mejor a peor.
    lienzo = Image.new("L", (limpia.width, sum(i.height + 10 for i in IMAGENES.values())), 255)
    y = 0
    for img in IMAGENES.values():
        lienzo.paste(img, (0, y)); y += img.height + 10
    display(lienzo)

**Qué ves** (con la semilla fija de la celda; en otro equipo pueden cambiar letras sueltas):
- **limpia (control):** 100 %, sin cambios. El montaje es inocente: los errores que siguen vienen de la **degradación**.
- **leve:** 99 % de parecido; solo cambia una mayúscula (`instalado:` → `Instalado:`). Inofensivo para BM25, que pasa todo a minúsculas.
- **media:** 98 %... y **un error que importa**: `Rojas` → `Fojas`. El "parecido" sigue altísimo, pero **el apellido del cliente ya no se puede buscar**. Este es el peligro del OCR: un 98 % suena a "casi perfecto" y el 2 % que falla puede ser justo el dato que preguntan.
- **fuerte:** **0 %: no devuelve nada** (y el log avisa *"RapidOCR returned empty result!"*). Con muy pocos píxeles, no hay texto que reconocer.

Eso es lo que muestra la última columna: **cuántas palabras** cambiaron y **cuáles**. Mide así, no con un solo porcentaje.

### Mitigación: ¿ayuda pre-procesar la imagen?

Dos arreglos baratos y comunes: **enderezar** (corregir la inclinación) y **ampliar** (más píxeles antes del OCR). Los probamos sobre `media` y `fuerte`.

In [ ]:
if not HAY_DOCLING:
    print(f"⚠️ Celda omitida: falta Docling. Instálalo con `{COMANDO_INSTALAR}` y reinicia el kernel (el resto del notebook sigue funcionando).")
else:
    # ¿Ayuda pre-procesar la imagen? Probamos dos arreglos baratos sobre los niveles "media" y "fuerte".
    # (En la vida real no conoces el ángulo exacto: se estima. Aquí lo conocemos para aislar el efecto.)
    ANGULO = {"media": 3, "fuerte": 4}

    def enderezar(img, grados):
        return img.rotate(-grados, expand=True, fillcolor=255, resample=Image.BICUBIC)

    def ampliar(img, factor=3):
        return img.resize((img.width * factor, img.height * factor), Image.LANCZOS)

    print(f"{'nivel':<8}{'tratamiento':<22}{'parecido':>9}   palabras que cambiaron")
    for nivel in ("media", "fuerte"):
        base = IMAGENES[nivel]
        tratamientos = {"sin tratar": base, "ampliar ×3": ampliar(base), "enderezar": enderezar(base, ANGULO[nivel]),
                        "enderezar + ampliar": ampliar(enderezar(base, ANGULO[nivel]))}
        for nombre, img in tratamientos.items():
            texto, _ = ocr_de_imagen(img, f"{nivel}_{nombre.replace(' ', '_').replace('×', 'x').replace('+', 'mas')}")
            parecido = difflib.SequenceMatcher(None, BUENO, texto).ratio()
            cambios = "; ".join(diferencias(BUENO, texto)) or "—"
            if parecido < 0.5 and texto:
                cambios = f"texto ilegible: «{texto[:48]}»"
            print(f"{nivel:<8}{nombre:<22}{parecido:>8.0%}   {cambios[:100]}")

    # Y la consecuencia para el RAG: ¿se puede encontrar al cliente por su apellido?
    print("\n¿Aparece 'Rojas' en el texto del OCR?")
    for nivel, texto in RESULTADOS_OCR.items():
        print(f"   {nivel:<18} {'sí' if 'rojas' in texto.lower() else 'NO'}")

**Qué ves** (y por qué hay que ser honesto con esto):
- **Ampliar no es una receta.** En `media` el `Rojas → Fojas` sigue ahí (ampliar la imagen **no recupera** los detalles que se perdieron al escanear) y aparecen errores **nuevos** (un punto en lugar de dos puntos).
- **Enderezar ayudó con el apellido** (`Rojas` aparece bien), pero cambió el error de sitio: ahora `Familiar` salió como `Famáliar`. Cada corrección **cambia** los errores; no los elimina.
- **En `fuerte`, nada lo arregla.** Sin tratar: vacío. Ampliando o enderezando, el OCR sí devuelve algo, pero es **texto ilegible** (parecido de 10 % y 27 %): peor que un vacío, porque **parece texto** y entra al índice. Con las dos correcciones juntas, vuelve a quedar vacío.
- **La consecuencia para el RAG:** `Rojas` se pierde desde el nivel `media`. Una búsqueda por el apellido del cliente fallaría, y nadie recibiría un error.

> **Regla práctica.** Lo que se perdió al escanear **no lo recupera ningún OCR**. Mejora la fuente (escanear a más resolución) antes que el pre-proceso; y si pre-procesas, **mide con tu documento**: no hay un tratamiento que siempre gane.

**Qué hacer con un texto de OCR en un RAG real:**
1. **Registra** que el texto salió de OCR (un metadato) y que la página casi sin texto se avisó.
2. **Conserva el original** como fuente de la cita, para que una persona verifique cifras y nombres.
3. **Valida** los datos críticos (códigos, importes, nombres) contra otra fuente cuando se pueda.
4. **Sospecha** de los vacíos y de los textos raros: cuenta caracteres y palabras "reales" por página.
5. Y recuerda: con **BM25** un solo carácter cambiado vuelve invisible un código; la búsqueda semántica tolera más, pero tampoco lo arregla.

### 2.5.4 Un solo parser para todos los formatos

Los cuatro lectores de `ingesta.py` (Markdown, HTML, DOCX y PDF) hacen cada uno lo suyo. Con Docling, **el mismo `DocumentConverter`** lee los cinco archivos de `datos/`. Mira cuántos encabezados y tablas encuentra en cada uno, si limpia el ruido de la página web y cómo entrega la tabla del Word.

In [ ]:
if not HAY_DOCLING:
    print(f"⚠️ Celda omitida: falta Docling. Instálalo con `{COMANDO_INSTALAR}` y reinicia el kernel (el resto del notebook sigue funcionando).")
else:
    from parsing_docling import convertidor

    ARCHIVOS = ["manual_soporte.md", "guia_tecnica.html", "condiciones_servicio.docx",
                "boletin_mantenimiento.pdf", "acta_escaneada.pdf"]
    conversor = convertidor(True)                     # UN solo objeto para los cinco archivos
    resultados_dl = {}
    print(f"{'archivo':<28}{'formato':<8}{'págs':>5}{'encabezados':>13}{'tablas':>8}{'seg':>7}")
    for nombre in ARCHIVOS:
        t0 = time.perf_counter()
        doc_dl = conversor.convert(f"datos/{nombre}").document
        seg = time.perf_counter() - t0
        resultados_dl[nombre] = doc_dl
        encabezados = sum(1 for it, _ in doc_dl.iterate_items() if str(it.label).endswith(("section_header", "title")))
        print(f"{nombre:<28}{nombre.split('.')[-1]:<8}{len(doc_dl.pages):>5}{encabezados:>13}{len(doc_dl.tables):>8}{seg:>7.1f}")

    # ¿Qué hizo Docling con el "ruido" de la página web y con la tabla del Word?
    md_html = resultados_dl["guia_tecnica.html"].export_to_markdown()
    print("\nHTML → ¿quedó el pie 'Síguenos'?", "Síguenos" in md_html, "· ¿el script 'gtag'?", "gtag" in md_html,
          "· ¿el menú 'Inicio'?", "Inicio" in md_html)
    print("\nDOCX → la tabla de penalizaciones, como tabla:\n")
    print(resultados_dl["condiciones_servicio.docx"].tables[0].export_to_markdown(doc=resultados_dl["condiciones_servicio.docx"]))

    # Y los cuatro lectores a mano, en la misma comparación: tiempo total con los lectores de ingesta.py.
    t0 = time.perf_counter()
    leer_carpeta(DATOS)
    print(f"\nLos 4 lectores a mano leyeron la carpeta en {(time.perf_counter() - t0) * 1000:.0f} ms (el escaneo, sin texto).")

**Qué ves.** El mismo objeto lee los cinco archivos. En Markdown, HTML y DOCX **no hay páginas**: el formato no las tiene, y por eso esos tres archivos tardan centésimas de segundo; los PDF tardan más porque pasan por el análisis de layout (y el acta, además, por el OCR). Docling **descartó solo** el menú, el script y el pie de la página web, y entregó la tabla del Word como tabla. Fíjate en la última línea: los lectores a mano son **más rápidos** (milisegundos) porque no hacen nada de visión.

### ¿Cuál uso? Parser a mano o Docling

| Situación | Lectores a mano (`ingesta.py`) | Docling |
| --- | --- | --- |
| Markdown, HTML o DOCX **sencillos** | ✅ Suficiente: rápidos, sin dependencias pesadas | Funciona, pero no aporta mucho |
| PDF con **tablas o columnas** | ❌ Tablas aplanadas, orden de lectura incierto | ✅ Tablas con estructura y orden de lectura |
| PDF **escaneado** | ❌ Solo avisan | ✅ OCR incluido (con sus errores) |
| Necesitas **citar con página** | Una "sección" por página (solo PDF) | ✅ Página y encabezados de cada fragmento |
| Entorno **ligero** (sin PyTorch ni modelos) | ✅ | ❌ Cientos de MB y CPU |
| Control fino de **cómo** se limpia una fuente concreta | ✅ Es tu código | Se configura, pero hay menos control |

**Cómo elegir, en una línea:** empieza con lo simple; pasa a Docling cuando midas que las **tablas, el orden de lectura o los escaneos** te cuestan respuestas.

> **Reto.** Escribe una función que recorra una carpeta, lea cada PDF **sin OCR** y reintente **con OCR** solo si el texto es casi vacío, y que guarde en un metadato si se usó OCR (la tienes hecha en `parsing_docling.leer_pdf_adaptativo`: ¿qué cambiarías?).

---
# Parte 3 · Estrategias de chunking

## ¿Por qué importa tanto?

El fragmento es la **unidad de búsqueda** y la **unidad de contexto**: es lo que se vectoriza (y se compara con la pregunta) y lo que se copia al prompt (y cuesta tokens). Por eso tiene dos presiones opuestas:

| Si es… | Al **buscar** | Al **responder** |
| --- | --- | --- |
| **Pequeño** | Vector preciso (un solo tema) | Poco contexto: la respuesta puede quedar partida entre dos fragmentos |
| **Grande** | Vector "diluido" (mezcla varios temas) | Contexto completo, pero **más tokens (dinero)** y más ruido |

No hay un tamaño correcto universal: **se mide**.

## Las estrategias

| Estrategia | Cómo parte | Idea / cuándo |
| --- | --- | --- |
| `fijo` | Ventanas de N palabras, sin mirar la gramática | La más simple y ciega: **corta ideas por la mitad** |
| `oraciones` | Junta oraciones hasta N palabras y repite las últimas (solape) | Buen punto de partida para texto corrido |
| `seccion` | Un fragmento por sección (o por página en un PDF) | Respeta la estructura; las secciones largas **diluyen** |
| `recursivo` | Corta por el separador **más grande que quepa** (párrafo → línea → oración → palabra) y junta vecinos hasta N palabras | La más usada en la práctica (es la idea del *RecursiveCharacterTextSplitter* de LangChain) |
| `semantico` | Mide qué tan parecidas son las **oraciones vecinas** (embeddings) y corta donde el tema cambia | Fragmentos coherentes; **cuesta una vectorización por oración** |
| `padre-hijo` | Se **busca** con hijos pequeños y se **entrega** el padre (la sección) | Precisión al buscar + contexto al responder |

**Metadatos:** cada fragmento guarda `fuente`, `seccion`, `n`, `tipo` y `publico`. Sin ellos no puedes **citar** ni **filtrar**.

> 🆕 En **3.4** verás una opción más, del parser Docling: el **chunking híbrido** (jerarquía de encabezados + límite de tokens). El laboratorio (Parte 4) la compara con las demás.

### Chunking, aterrizado

**🧠 Qué es.** Partir cada sección en **fragmentos** (*chunks*). Un fragmento cumple **dos roles a la vez**: es la **unidad de búsqueda** (lo que se vectoriza y se compara con la pregunta) y la **unidad de contexto** (lo que se copia al prompt y se paga en tokens).

**⚙️ Qué hace.** Decide **dónde cortar** y **qué repetir** (el solape) para que cada trozo sea una idea completa.

**🎯 Para qué sirve.** Es la decisión que más mueve la calidad después de la ingesta: define qué compite en la búsqueda y cuánto texto llega al modelo.

**🏗️ Analogía de infraestructura.** Piensa en cómo partes **logs multilínea**. Si cortas cada N bytes sin mirar el formato, un *stack trace* queda a la mitad y ninguna de las dos mitades sirve; si agrupas por evento, cada registro es legible. Y también está el tamaño: muy pequeño, pierdes el contexto de la excepción; muy grande, el evento mezcla cosas que no tienen que ver.

**✏️ Ejemplo a mano: las dos presiones.** Una sección de 200 palabras habla de tres temas: precio, instalación y garantía.
- Como **un solo fragmento**: su vector es una "mezcla" de los tres temas. La pregunta "¿cuánto cuesta?" se compara contra esa mezcla y la señal del precio queda diluida. Pero si la respuesta es "39 dólares, incluida la instalación", **todo está en el mismo trozo**.
- Como **tres fragmentos de ≈65 palabras**: cada vector habla de un solo tema, la búsqueda es más precisa; pero la respuesta que cruza precio e instalación puede quedar **partida entre dos trozos**, y el prompt solo recibe uno.
- El costo: en nuestro laboratorio, enviar secciones enteras cuesta ≈ 345 palabras de contexto por pregunta frente a ≈ 160 con fragmentos de 70 palabras (más del doble). Por eso un MRR más alto de `seccion` no sale gratis.

**⚠️ Cuándo no complicarse.** Con documentos cortos y homogéneos (una FAQ de una línea por pregunta) cada entrada **ya es** un fragmento: partirla más solo daña.

**🐞 Errores típicos.** (1) Elegir el tamaño "por costumbre". (2) Cortar por caracteres y partir un número de su unidad. (3) Poner **mucho solape** (duplica texto y costo) o ninguno. (4) Olvidar los **metadatos**: sin fuente y sección no se puede citar ni filtrar.

### Las seis estrategias, una por una

| Estrategia | 🧠 Qué es | ⚙️ Cómo corta | 🏗️ Analogía | ⚠️ Cuándo no |
| --- | --- | --- | --- | --- |
| **`fijo`** | Ventana de N palabras | Avanza N menos un solape de 10 palabras (con `solape=1`), sin mirar la gramática | `split -b`: corta por bytes, sin respetar líneas | Casi siempre: parte frases y datos. Solo para texto sin estructura alguna |
| **`oraciones`** | Junta oraciones completas | Va sumando oraciones hasta N palabras; repite las últimas `solape` oraciones en el siguiente | Agrupar líneas de log por evento | Cuando una oración sola ya excede N, o el texto no tiene puntuación |
| **`seccion`** | Un fragmento por sección o página | No corta: usa la estructura del documento | Un runbook completo por archivo | Secciones largas: diluyen el vector y cuestan contexto |
| **`recursivo`** | Corta por el separador **más grande que quepa** | Prueba párrafo (`\n\n`), luego línea, luego oración, luego palabra; junta vecinos hasta N | Un `logrotate` que respeta los límites naturales y solo baja de nivel si no cabe | Cuando no vale la pena la complejidad: con texto limpio da casi lo mismo que `oraciones` |
| **`semantico`** | Corta donde **cambia el tema** | Vectoriza cada oración, mide el coseno entre vecinas y corta en el 25 % de uniones más flojas (con tope de N palabras) | Detectar cambio de servicio en un log mezclado | Cuando importa el costo de ingesta (vectoriza **cada oración**) o los resultados no mejoran |
| **`padre-hijo`** | Busca con trozos chicos, entrega el grande | Hijos de ≈35 palabras por oración; cada uno recuerda su sección (el padre) | Buscar por la alerta y abrir el runbook completo | Cuando los padres son enormes: el LLM recibe mucho que no necesita |

**✏️ Ejemplo a mano.** El texto *"El pago vence el día 15. Si no se paga hasta el día 15, se cobra un recargo de 3 dólares. Si sigue sin pagar el día 25, el servicio se suspende."* con ventanas de 18 palabras:
- `fijo` puede terminar en "… el día 25, el servicio se" (**mitad de frase**).
- `oraciones` y `recursivo` terminan en un punto.
- `padre-hijo`: busca por "recargo de 3 dólares" y entrega **toda la sección** de pagos.

Ejecútalo en la celda siguiente (no consume nada):

In [ ]:
# Ejemplo a mano: la misma sección de juguete, partida de varias maneras (sin LLM, sin costo)
from ingesta import DocumentoExt
texto = ("El pago vence el día 15 de cada mes. Si no se paga hasta el día 15, se cobra un recargo de 3 dólares. "
         "Si sigue sin pagar el día 25, el servicio se suspende. Para reactivarlo hay que pagar la deuda y llevar la cédula a una agencia.")
doc = DocumentoExt(fuente="juguete.md", secciones=[("Pagos", texto)], tipo="md")
for estrategia in ("fijo", "oraciones", "recursivo"):
    frs = fragmentar_doc(doc, estrategia=estrategia, max_palabras=18, solape=1)
    print(f"━━ {estrategia} ({len(frs)} fragmentos)")
    for f in frs:
        print("   ·", f.texto)
hijos_demo = fragmentar_padre_hijo(doc, 12)
print(f"━━ padre-hijo: {len(hijos_demo)} hijos; todos comparten un padre de {len(hijos_demo[0].metadatos['texto_padre'].split())} palabras")
for h in hijos_demo:
    print("   ·", h.id, "|", h.texto)

## 3.1 Una misma sección, cuatro maneras de partirla

Veamos la sección **"3. Facturación y pagos"** del manual con fragmentos de ≈40 palabras. Mira **dónde termina cada fragmento**.

In [ ]:
manual = next(d for d in documentos if d.fuente == "manual_soporte.md")
facturacion = type(manual)(fuente=manual.fuente, secciones=[s for s in manual.secciones if s[0].startswith("3.")], tipo="md")

def mostrar(estrategia, tam=40):
    frs = fragmentar_doc(facturacion, estrategia=estrategia, max_palabras=tam)
    print(f"━━ {estrategia} ({tam} palabras) → {len(frs)} fragmentos")
    for f in frs:
        p = f.texto.split()
        print(f"   [{f.metadatos['n']}] ({len(p):>2} pal.)  {' '.join(p[:5])} … {' '.join(p[-5:])}")
    print()

for estrategia in ("fijo", "oraciones", "recursivo"):
    mostrar(estrategia)

**Observa:** con `fijo`, los fragmentos **terminan a mitad de frase** (por ejemplo, un número separado de su unidad). Con `oraciones` y `recursivo`, terminan en un punto. Esa diferencia, que parece cosmética, hace que un fragmento sea **leíble** y su vector **coherente**.

## 3.2 Fragmentado semántico: cortar donde cambia el tema

El fragmentado semántico vectoriza **cada oración**, calcula la similitud coseno entre oraciones consecutivas y **corta donde cae más** (las uniones más "flojas"). Así se ve en la sección de *Fallas y soporte técnico*:

In [ ]:
import numpy as np
from embeddings import vectorizar
from rag import _oraciones

fallas = next(t for titulo, t in manual.secciones if titulo.startswith("5."))
oraciones = _oraciones(fallas)
v = np.array(vectorizar(oraciones, tipo="documento"), dtype=np.float32)
v /= np.linalg.norm(v, axis=1, keepdims=True)
similitud = (v[:-1] * v[1:]).sum(axis=1)
corte = np.percentile(similitud, 25)
print(f"{len(oraciones)} oraciones · se cortaría donde la similitud con la siguiente es ≤ {corte:.2f}\n")
for i, s in enumerate(similitud):
    marca = "✂️  CORTE" if s <= corte else ""
    print(f"  {s:.2f}  {oraciones[i][:62]:<62}… {marca}")

Las uniones marcadas son los **cambios de tema** que detectó el modelo. Es elegante, pero **no siempre mejora los resultados** (lo mediremos en la Parte 4) y cuesta más tiempo de cómputo en la ingesta: vectoriza cada oración, no solo cada fragmento.

## 3.3 ¿Cuántos fragmentos salen de cada una?

In [ ]:
print(f"{'estrategia':<11}{'palabras':>9}{'fragmentos':>12}{'mínimo':>8}{'media':>8}{'máximo':>8}")
for estrategia in ESTRATEGIAS:
    for tam in ([70] if estrategia == "seccion" else (40, 70, 120)):
        r = resumen_fragmentos(fragmentar_corpus(documentos, estrategia=estrategia, max_palabras=tam))
        print(f"{estrategia:<11}{('—' if estrategia == 'seccion' else tam):>9}{r['n']:>12}{r['min']:>8}{r['media']:>8}{r['max']:>8}")

`seccion` produce **pocos fragmentos grandes** (hasta ≈200 palabras): es lo que pagarás en el prompt. `semantico` produce **muchos fragmentos de tamaño desigual**. En la siguiente parte vemos qué le pasa a la calidad.

## 3.4 Chunking híbrido con Docling

Las estrategias anteriores parten **texto**. Docling parte un **documento con estructura**: ya sabe qué es cada título, párrafo, lista y tabla. Su `HybridChunker` combina dos ideas en dos pasos:

1. **Jerarquía.** Parte de los elementos del documento (un párrafo, un ítem de lista, una **tabla entera**) y cada fragmento **lleva los encabezados** que lo contienen (`Parte B · Clientes Negocio › Compensaciones`) y las **páginas** donde está.
2. **Tokens.** Lo que **pasa del máximo** de tokens se parte, y los vecinos **muy pequeños** con los mismos encabezados se **unen** hasta llenar el espacio.

> ### ⚠️ No lo confundas: "híbrido" NO es la búsqueda híbrida
> | | **Chunking híbrido** (Docling) | **Búsqueda híbrida** (Parte 6) |
> | --- | --- | --- |
> | Qué decide | **Dónde cortar** el documento | **Cómo buscar** |
> | Se combina | Jerarquía del documento + límite de tokens | BM25 + vectores (con RRF) |
> | Cuándo ocurre | En la **ingesta** (una vez) | En la **consulta** (en cada pregunta) |
> | Dónde se mide | Parte 4 (laboratorio) | Parte 6 |
>
> Comparten la palabra "híbrido" y nada más. Se pueden usar **las dos a la vez**.

### Tres detalles que hay que entender

- **`contextualize(fragmento)`** devuelve **encabezados + texto**: es lo que conviene **vectorizar** (un fragmento suelto como "Si la caída supera las 4 horas…" no dice a quién aplica; con su encabezado, sí). El `.text` a secas es el texto sin encabezados.
- **El tokenizador debe ser el del modelo de embeddings.** Los **tokens** no son palabras, y cada modelo los cuenta a su manera. Si el chunker cuenta con otro tokenizador, "128 tokens" puede ser más (o menos) para el modelo real. Aquí usamos el de `nomic-embed-text-v2-moe`, el mismo de los embeddings de la sesión.
- **Las tablas no se parten por la mitad** si caben: la tabla del boletín queda entera, con sus filas, dentro de un fragmento.

### El chunking híbrido, aterrizado

**🧠 Qué es.** Un fragmentador que usa la **estructura** del documento (jerarquía de encabezados) y un **límite de tokens** al mismo tiempo. No cruza los límites de los elementos, no parte tablas que caben, y no deja fragmentos diminutos sueltos.

**⚙️ Qué hace.** (1) Genera un fragmento por elemento del documento, con sus encabezados y páginas. (2) **Parte** los que exceden `max_tokens`. (3) **Une** los vecinos pequeños que comparten encabezados mientras quepan. Cada fragmento conserva `meta.headings` y las páginas de sus elementos.

**🎯 Para qué sirve.** Para obtener fragmentos **coherentes** (respetan la estructura), de **tamaño acotado** (caben en el modelo de embeddings) y **citables** (traen encabezados y página), sin escribir un divisor propio.

**🏗️ Analogía de infraestructura.** Es el **empaquetado de logs por evento con un tamaño máximo de lote**: primero respetas los límites naturales (un evento completo, un stack trace entero), y luego aplicas el tamaño: el evento enorme se trocea, y los eventos diminutos del mismo servicio se agrupan en un lote hasta llenarlo.

**✏️ Ejemplo a mano.** La celda siguiente aplica la idea con "tokens" de juguete (una palabra = un token) y un máximo de 14. Dos párrafos cortos de *Pagos* se **unen** (13 tokens); el párrafo largo se **parte** en dos; *Garantía* queda **aparte** porque tiene otro encabezado, aunque quepa. (Docling parte con un divisor que respeta frases cuando puede; el ejemplo parte por palabras para que se vea el mecanismo.)

**⚠️ Cuándo no.** Si tus documentos no tienen estructura (un texto plano sin títulos), la jerarquía no aporta y el resultado se parece a un recursivo por tokens. Y con documentos ya cortos y autónomos, partirlos más solo daña. Requiere Docling (dependencia pesada), y **cuenta tokens con el tokenizador que le des**: si le das el equivocado, el límite engaña.

**🐞 Errores típicos.** (1) Confundirlo con la **búsqueda híbrida** (recuadro de arriba). (2) Usar un tokenizador distinto al del modelo de embeddings. (3) Vectorizar solo `.text` y perder el contexto de los encabezados (usa `contextualize`, o antepón la sección como hace `rag.texto_a_vectorizar`). (4) Pensar que `max_tokens` cuenta solo el texto: **cuenta también los encabezados**. (5) Fijar un máximo muy bajo (64) que parte lo que debía ir junto.

In [ ]:
# Ejemplo a mano (sin Docling): el chunking híbrido en miniatura, con "tokens" de juguete (1 palabra = 1 token)
MAXIMO = 14
elementos = [   # (encabezado, texto): cada elemento ya es un párrafo, un ítem de lista o una tabla
    ("Pagos", "El pago vence el día 15."),
    ("Pagos", "Hay un recargo de 3 dólares."),
    ("Pagos", "El servicio se suspende el día 25 si sigue sin pagar, y para reactivarlo hay que pagar la deuda y llevar la cédula a una agencia."),
    ("Garantía", "Cubre defectos de fabricación."),
]

def tokens(encabezado, texto):                          # el límite cuenta TAMBIÉN el encabezado
    return len((encabezado + " " + texto).split())

# 1) Jerarquía: cada elemento nace como un fragmento, con el encabezado que lo contiene.
# 2) Tokens: lo que pasa del máximo se parte...
partidos = []
for enc, txt in elementos:
    palabras = txt.split()
    cupo = MAXIMO - len(enc.split())
    for i in range(0, len(palabras), cupo):
        partidos.append((enc, " ".join(palabras[i:i + cupo])))

# 3) ...y los vecinos pequeños con el MISMO encabezado se unen mientras quepan. Nunca se une "Pagos" con "Garantía".
fragmentos = []
for enc, txt in partidos:
    if fragmentos and fragmentos[-1][0] == enc and tokens(enc, fragmentos[-1][1] + " " + txt) <= MAXIMO:
        fragmentos[-1] = (enc, fragmentos[-1][1] + " " + txt)
    else:
        fragmentos.append((enc, txt))

for n, (enc, txt) in enumerate(fragmentos):
    print(f"[{n}] {tokens(enc, txt):>2} tokens · {enc} | {txt}")

In [ ]:
if not HAY_DOCLING:
    print(f"⚠️ Celda omitida: falta Docling. Instálalo con `{COMANDO_INSTALAR}` y reinicia el kernel (el resto del notebook sigue funcionando).")
else:
    from parsing_docling import crear_chunker, leer_con_docling, paginas_de, tokenizador_para

    TOKENIZADOR = tokenizador_para(modelo_embeddings())
    print("Modelo de embeddings:", modelo_embeddings())
    print("Tokenizador que usa el chunker:", TOKENIZADOR, "\n")
    chunker = crear_chunker(max_tokens=128, tokenizador=TOKENIZADOR)

    for archivo in ("condiciones_servicio.docx", "boletin_mantenimiento.pdf"):
        trozos = list(chunker.chunk(dl_doc=leer_con_docling(f"datos/{archivo}")))
        print(f"━━ {archivo}: {len(trozos)} fragmentos (como máximo 128 tokens, contando los encabezados)")
        for i, c in enumerate(trozos):
            tokens = chunker.tokenizer.count_tokens(chunker.contextualize(c))
            paginas = paginas_de(c) or "—"
            print(f"  [{i}] {tokens:>3} tok · págs {str(paginas):<4} · {' › '.join((c.meta.headings or ['(sin encabezado)'])[-2:])[:84]}")
        print()

    # contextualize(): lo que conviene VECTORIZAR (encabezados + texto) frente a lo que es solo el texto.
    trozos_docx = list(chunker.chunk(dl_doc=leer_con_docling("datos/condiciones_servicio.docx")))
    c = trozos_docx[6]
    print("c.text           →", c.text[:90], "…")
    print("contextualize(c) →", chunker.contextualize(c)[:190].replace("\n", " | "), "…")

    # Una tabla se conserva entera (y con sus filas) dentro de UN fragmento.
    c_tabla = next(c for c in chunker.chunk(dl_doc=leer_con_docling("datos/boletin_mantenimiento.pdf")) if "Zona Sur" in c.text)
    print("\nEl fragmento de la tabla del boletín:\n" + c_tabla.text)

In [ ]:
if not HAY_DOCLING:
    print(f"⚠️ Celda omitida: falta Docling. Instálalo con `{COMANDO_INSTALAR}` y reinicia el kernel (el resto del notebook sigue funcionando).")
else:
    import statistics
    from rag import texto_a_vectorizar

    docs_dl = [leer_con_docling(f"datos/{n}") for n in ARCHIVOS]
    print(f"{'max_tokens':>11}{'fragmentos':>12}{'mín':>6}{'media':>7}{'máx':>6}   (tokens contando los encabezados; los cinco documentos)")
    for maximo in (64, 128, 256):
        ch = crear_chunker(maximo, TOKENIZADOR)
        tokens = [ch.tokenizer.count_tokens(ch.contextualize(c)) for d in docs_dl for c in ch.chunk(dl_doc=d)]
        print(f"{maximo:>11}{len(tokens):>12}{min(tokens):>6}{statistics.mean(tokens):>7.0f}{max(tokens):>6}")

    # Palabras y tokens no son lo mismo: ¿cuántos tokens tienen los fragmentos de "oraciones de 70 palabras"?
    frs_70 = fragmentar_corpus(documentos, estrategia="oraciones", max_palabras=70)
    tok_70 = [chunker.tokenizer.count_tokens(texto_a_vectorizar(f)) for f in frs_70]
    pal_70 = [len(texto_a_vectorizar(f).split()) for f in frs_70]
    print(f"\nOraciones de 70 palabras (con su sección): {statistics.mean(pal_70):.0f} palabras ≈ {statistics.mean(tok_70):.0f} tokens "
          f"de media (máx. {max(tok_70)}) con este tokenizador.")

**Qué ves.**
- **Jerarquía y páginas.** El Word da 10 fragmentos, cada uno con sus encabezados (`Parte A · Clientes residenciales › Compensaciones`) y **sin páginas** (el DOCX no las tiene). El PDF da 5 fragmentos, cada uno con su **página**; la tabla y el párrafo que la sigue quedan en fragmentos distintos ([1] y [2]), los dos bajo "Ventanas por zona", sin partir la tabla.
- **`contextualize`.** Añade los encabezados al texto: así se vectoriza *"…Parte B · Clientes Negocio | Compensaciones | Si la caída supera las 4 horas…"* y se sabe a quién aplica.
- **La tabla del boletín** queda **entera, con sus filas**, en un solo fragmento (103 tokens).
- **Tamaños en tokens.** Con `max_tokens=128`: 55 fragmentos de **38 a 126 tokens** (media 81). Con 64: 106 fragmentos; con 256: 37. El máximo se respeta (cuenta los encabezados): 126 de 128, 64 de 64, 253 de 256.
- **Palabras contra tokens.** Los fragmentos de "oraciones de 70 palabras" tienen en media **77 tokens** con este tokenizador (máximo 140): son del mismo orden que los de 128 tokens de Docling (media 81). Por eso, más abajo, su contexto va a ser parecido.

---
# Parte 4 · Laboratorio de chunking

Probamos **todas las combinaciones** de estrategia y tamaño con la **misma búsqueda** (semántica) y las mismas 22 preguntas base. Para cada una: fragmentar → vectorizar (con caché) → medir.

> ⏱️ La primera vez tarda ≈ 20 a 40 segundos con el modelo local (vectoriza todo el corpus varias veces). Las siguientes veces la caché lo hace casi instantáneo.

In [ ]:
def probar(estrategia, tam, modo="vectorial", preguntas=BASE, k=3):
    """Fragmenta el corpus con una configuración, arma el índice y mide."""
    frs = fragmentar_corpus(documentos, estrategia=estrategia, max_palabras=tam)
    ix = construir_indice(frs)
    r = evaluar_busqueda(preguntas, lambda p, k: MODOS[modo](ix, p["pregunta"], k, p.get("filtro")), k)
    r["fragmentos"] = len(frs)
    return r

if HAY_EMB:
    t0 = time.perf_counter()
    resultados = []
    for estrategia in ESTRATEGIAS:
        for tam in ([70] if estrategia == "seccion" else (40, 70, 120)):
            r = probar(estrategia, tam)
            r.update(estrategia=estrategia, tam="—" if estrategia == "seccion" else tam)
            resultados.append(r)
    resultados.sort(key=lambda r: (-r["mrr"], r["contexto"]))
    print(f"({time.perf_counter() - t0:.0f} s)\n")
    print(f"{'estrategia':<11}{'palabras':>9}{'fragm.':>8}{'hit@1':>8}{'hit@3':>8}{'MRR':>7}{'contexto':>10}")
    for r in resultados:
        print(f"{r['estrategia']:<11}{r['tam']:>9}{r['fragmentos']:>8}{r['hit1']:>5}/{r['n']:<2}{r['hitk']:>5}/{r['n']:<2}{r['mrr']:>7.2f}{r['contexto']:>10.0f}")

## Cómo leer esta tabla (la parte importante)

1. **Mira la columna *contexto* antes de coronar a un ganador.** `seccion` suele quedar arriba en MRR: la sección entera casi siempre **contiene** la respuesta. Pero entrega **2 o 3 veces más palabras** al LLM que los fragmentos de 40 a 70 palabras: más tokens, más dinero y más ruido. Un fragmento grande "gana" en esta métrica **porque es grande**.
2. **`fijo` con 40 palabras** es el peor: sin respetar la gramática y con ventanas pequeñas, corta datos por la mitad.
3. **`oraciones`, `recursivo` y `semantico` quedan muy parejos.** Con 22 preguntas, 1 o 2 aciertos de diferencia son **ruido**: no declares ganador a uno por una pregunta.
4. **No hay ganador universal.** Cambia el modelo de embeddings, los documentos o las preguntas y la tabla cambia. **Lo valioso es el método**: probar, medir y decidir con números (la misma disciplina de la Sesión 3, aplicada a la recuperación).

> 📊 **Resultados de referencia del instructor** (modelo `nomic-embed-text-v2-moe`, búsqueda semántica): `seccion` ≈ 20/22 en hit@1 con ≈ 350 palabras de contexto; `oraciones` de 70 palabras ≈ 18/22 con ≈ 160 palabras; `fijo` de 40 palabras ≈ 13/22. Tus números pueden diferir en 1 o 2 unidades.

## 4.1 El patrón padre-hijo: lo mejor de los dos mundos

Si los fragmentos **pequeños** buscan mejor pero los **grandes** responden mejor, ¿por qué no usar ambos?

```
INGESTA     sección (padre) ─► se parte en hijos de ≈35 palabras ─► se vectorizan los HIJOS
CONSULTA    pregunta ─► se buscan los hijos más parecidos ─► se entrega el PADRE de cada uno (sin repetir)
```

Es como un **índice de libro**: buscas por la entrada precisa, pero lees la página completa.

In [ ]:
if HAY_EMB:
    hijos = [h for d in documentos for h in fragmentar_padre_hijo(d, 35)]
    ix_ph = construir_indice(hijos)
    print("Hijos:", resumen_fragmentos(hijos))
    ej = hijos[40]
    print("\nUn hijo:     ", ej.texto[:100], "…")
    print("Su padre es: ", ej.metadatos["padre_id"], f"({len(ej.metadatos['texto_padre'].split())} palabras)\n")

    linea("Oraciones 70 (línea base)", evaluar_busqueda(BASE, lambda p, k: buscar_vectorial(indice_base, p["pregunta"], k), 3))
    linea("Padre-hijo (hijos de 35)",
          evaluar_busqueda(BASE, lambda p, k: expandir_a_padre(buscar_vectorial(ix_ph, p["pregunta"], 10))[:k], 3))

**Qué verás:** el padre-hijo iguala o supera a la sección entera en hit@1 y MRR, pero **también entrega secciones completas** (contexto grande). Es la técnica correcta cuando tu documento tiene secciones coherentes y quieres **citar la sección completa**; es mala si las secciones son enormes (el LLM recibe mucho que no necesita).

### Padre-hijo, aterrizado

**🧠 Qué es.** Un patrón con **dos niveles**: los **hijos** (trozos pequeños, ≈35 palabras) se vectorizan y se buscan; el **padre** (la sección que los contiene) es lo que se **entrega** al LLM.

**⚙️ Qué hace.** En la ingesta, cada hijo guarda en sus metadatos el `padre_id` y el `texto_padre`. En la consulta se buscan los hijos más parecidos, se **reemplaza cada hijo por su padre** y se **eliminan duplicados** (si tres hijos son de la misma sección, el padre se envía una sola vez).

**🎯 Para qué sirve.** Para resolver la tensión de las dos presiones: **precisión al buscar** (vectores de un solo tema) y **contexto completo al responder** (la sección entera).

**🏗️ Analogía de infraestructura.** Una **alerta** apunta a su **runbook**. Buscas por la alerta (precisa, corta) pero lo que lees para actuar es el runbook completo. También es el **índice de un libro**: buscas la entrada, lees la página.

**✏️ Ejemplo a mano.** Sección de pagos (49 palabras) → 4 hijos (de 9 a 14 palabras, uno por oración). La pregunta "¿cuánto es el recargo?" coincide fuerte con el hijo 2 ("…se cobra un recargo de 3 dólares"). Se entrega el padre completo: incluye también cuándo se suspende y cómo reactivar. Si el hijo 2 y el hijo 3 salen ambos entre los 10 primeros, el padre se envía **una vez**.

**⚠️ Cuándo no.** Si tus secciones son **gigantes** (un libro sin títulos de 3 000 palabras), el "padre" sería todo el libro y el LLM recibe ruido y costo. Antes hay que definir mejores padres (por ejemplo, por subtítulo o por página). Y con un documento de fragmentos ya cortos, no aporta.

**🐞 Errores típicos.** (1) Olvidar que el **contexto** sube (el laboratorio lo muestra: ≈ 364 palabras). (2) Buscar solo `k` hijos y no 10: al expandir a padres se repiten y quedan menos de `k`. (3) Padres mal formados (secciones mezcladas) que arrastran ruido.

## 4.2 `docling-hibrido` en el laboratorio

Ahora la pregunta de siempre: ¿**mejora** la recuperación? Añadimos una configuración nueva, **`docling-hibrido`**: los **cinco documentos leídos con Docling** (un solo parser, con OCR para el acta) y partidos con `HybridChunker`, con los **mismos metadatos** que las demás (`fuente`, `seccion`, `tipo`, `publico`) y más (`paginas`, `tokens`). La comparamos con las otras, con las **mismas 22 preguntas** y la misma búsqueda semántica. Probamos tres tamaños (64, 128 y 256 tokens). El tamaño de referencia es **128**, que es el que habíamos fijado antes de medir (se parece al de "oraciones de 70 palabras").

> ⏱️ Los PDF ya se convirtieron en 2.5 (Docling los recuerda), así que la celda tarda poco. Es el mismo laboratorio de `practica/laboratorio_chunking.py`, que ahora también acepta `docling-hibrido`.

In [ ]:
if not HAY_EMB:
    print("⚠️ Celda omitida: no hay servidor de embeddings (ver Parte 0).")
elif not HAY_DOCLING:
    print(f"⚠️ Celda omitida: falta Docling. Instálalo con `{COMANDO_INSTALAR}` y reinicia el kernel (el resto del notebook sigue funcionando).")
else:
    from parsing_docling import fragmentar_carpeta_hibrido

    t0 = time.perf_counter()
    filas = [probar(e, t) | {"tam": t if e != "seccion" else "—", "etiqueta": e}
             for e, t in (("fijo", 40), ("oraciones", 70), ("seccion", 70))]
    # padre-hijo, con las mismas líneas que en 4.1
    hijos_35 = [h for d in documentos for h in fragmentar_padre_hijo(d, 35)]
    ix_ph35 = construir_indice(hijos_35)
    r = evaluar_busqueda(BASE, lambda p, k: expandir_a_padre(buscar_vectorial(ix_ph35, p["pregunta"], 10))[:k], 3)
    filas.append(r | {"fragmentos": len(hijos_35), "tam": 35, "etiqueta": "padre-hijo"})

    indices_dl = {}
    for tokens in (64, 128, 256):
        frs = fragmentar_carpeta_hibrido(DATOS, max_tokens=tokens)       # los 5 documentos, leídos con Docling
        indices_dl[tokens] = construir_indice(frs)
        r = evaluar_busqueda(BASE, lambda p, k: buscar_vectorial(indices_dl[tokens], p["pregunta"], k), 3)
        filas.append(r | {"fragmentos": len(frs), "tam": f"{tokens} tok", "etiqueta": "docling-hibrido"})
    print(f"({time.perf_counter() - t0:.0f} s)\n")

    filas.sort(key=lambda r: (-r["mrr"], r["contexto"]))
    print(f"{'estrategia':<17}{'tamaño':>9}{'fragm.':>8}{'hit@1':>8}{'hit@3':>8}{'MRR':>7}{'contexto':>10}")
    for r in filas:
        print(f"{r['etiqueta']:<17}{r['tam']:>9}{r['fragmentos']:>8}{r['hit1']:>5}/{r['n']:<2}{r['hitk']:>5}/{r['n']:<2}{r['mrr']:>7.2f}{r['contexto']:>10.0f}")

    # Lo que Docling añade: cada resultado sabe de qué página y de qué encabezados viene (para CITAR).
    mejor_dl = indices_dl[128]
    pregunta_tabla = next(p for p in BASE if p["id"] == "P15")
    print(f"\nPregunta: {pregunta_tabla['pregunta']}")
    for r in buscar_vectorial(mejor_dl, pregunta_tabla["pregunta"], 2):
        m = r.metadatos
        marca = "✓ contiene la clave" if contiene_clave(r.texto, pregunta_tabla["clave"]) else "✗ no la contiene"
        print(f"   {m['fuente']} · págs {m['paginas']} · «{m['seccion']}» · {m['tokens']} tok · tipo={m['tipo']} · público={m['publico']}  {marca}")
    fallos_dl = next(r for r in filas if r["etiqueta"] == "docling-hibrido" and r["tam"] == "128 tok")["fallos"]
    print("\nPreguntas en las que docling-hibrido de 128 tokens NO quedó primero:",
          ", ".join(f"{f['id']} (puesto {f['posicion'] or '—'})" for f in fallos_dl))

**Cómo leerlo, con honestidad:**

- **`docling-hibrido` de 128 tokens: 18/22 en hit@1, 22/22 en hit@3, MRR 0.91, ≈ 158 palabras.** Frente a **oraciones de 70: 18/22, 21/22, MRR 0.87, ≈ 160**. Es un **empate técnico**: mismo hit@1, una pregunta más en hit@3 y cuatro centésimas de MRR, que con 22 preguntas son **ruido**. El contexto es casi igual porque los fragmentos tienen un tamaño parecido.
- **Variando el tamaño:** con **256 tokens** sube a 19/22 y MRR 0.92, pero entrega **≈ 243 palabras** (más contexto: recuerda que un fragmento grande acierta más por ser grande). Con **64 tokens** baja a 17/22 y MRR 0.84 (≈ 71 palabras).
- **`padre-hijo` y `seccion` siguen arriba** por la razón de siempre: entregan ≈ 345 a 364 palabras. Y `fijo` de 40 sigue siendo la peor.

**¿Entonces para qué sirve? No para ganar este examen.** Lo que aporta es otra cosa:
1. **Un solo parser** para todos los formatos, **con OCR**: el acta escaneada ya es buscable (ninguna de las 22 preguntas la usa, así que no se nota en la nota).
2. **La tabla del boletín entera**, con sus filas, en un fragmento.
3. **Citas con página y encabezados** exactos (mira las líneas de la pregunta de la tabla: cada resultado sabe de qué página y de qué sección viene).
4. **Tamaño controlado en tokens**, medido con el tokenizador del modelo de embeddings.

> ⚠️ **Una trampa de evaluación (medida).** La `clave` de la pregunta P16 es `6 | 6 | 90`, escrita con el **formato de tabla de nuestro lector del Word**. Con el formato por defecto de Docling para tablas (cada celda como una frase "columna = valor") esa clave no aparece y `docling-hibrido` baja a **16/22 y MRR 0.84**. En este laboratorio usamos tablas en Markdown (`tablas_markdown=True`), donde sí aparece. Moraleja: **una `clave` atada al formato de un lector** puede hacer perder a otro lector aunque su respuesta sea correcta. Al cambiar de parser, revisa también las claves.

**Reto:** corre `uv run python practica/laboratorio_chunking.py --estrategias oraciones,docling-hibrido --tokens 64,128,256 --modo hibrido` y compara con la búsqueda híbrida (BM25 + vectores). ¿Cambia la conclusión? (Lo medimos: con la búsqueda híbrida, `docling-hibrido` de 128 tokens dio 19/22 y MRR 0.91, contra 18/22 y 0.86 de `oraciones` de 70; sigue siendo una pregunta de diferencia, o sea, ruido.)

## Ejercicio

1. Cambia el tamaño de los hijos a 25 y a 50. ¿Cambia algo?
2. ¿Qué pasaría con un PDF de una sola sección de 3 000 palabras (un libro sin títulos)? ¿Qué sería el "padre"?
3. Corre la **práctica** (`practica/laboratorio_chunking.py`) con `--tipos lexica` y compara con `--tipos semantica`. ¿La mejor estrategia es la misma?

---
# Parte 5 · Búsqueda léxica: BM25

## El problema de la búsqueda semántica

Un modelo de embeddings **aprende significado**, y lo hace muy bien con paráfrasis. Pero los **identificadores** (`OPT-1107`, `E-217`, `0800-26337`, un número de factura, un SKU) casi no tienen significado: son **etiquetas exactas**. Para el modelo, `OPT-1102` y `OPT-1107` son casi el mismo texto.

## La solución clásica: BM25

**BM25** es el algoritmo de búsqueda por **palabras** que usan Elasticsearch, OpenSearch y Lucene desde hace décadas. Puntúa más a un fragmento cuando:

- contiene las palabras de la consulta,
- esas palabras son **raras** en el corpus (`OPT-1107` pesa mucho; "el", poco): es el **IDF**,
- aparecen varias veces, pero con **rendimientos decrecientes** (parámetro `k1`),
- y **sin favorecer a los fragmentos más largos** (parámetro `b`).

No entiende significado: "contraseña" y "clave" son palabras distintas. A cambio, **nunca se confunde con un código**.

### BM25, aterrizado

**🧠 Qué es.** Un algoritmo de **puntaje por palabras**: dada una consulta, le asigna un número a cada fragmento según qué palabras de la consulta contiene. Es el corazón de Lucene, Elasticsearch y OpenSearch.

**⚙️ Qué hace.** Suma, para cada palabra de la consulta que aparece en el fragmento, un aporte con tres ingredientes:

| Ingrediente | Idea | Parámetro |
| --- | --- | --- |
| **TF** (frecuencia del término) | Más apariciones, más puntaje, pero con **rendimientos decrecientes** | `k1 = 1.5` |
| **IDF** (rareza en el corpus) | Una palabra rara dice mucho; una común, casi nada | (calculado del corpus) |
| **Normalización por longitud** | No favorece a un fragmento solo por ser largo | `b = 0.75` |

**🎯 Para qué sirve.** Para lo que los embeddings hacen mal: **identificadores exactos** (`OPT-1107`, `E-217`, un SKU, un teléfono), nombres propios y cifras. Además es barato: no necesita modelo ni vectores.

**🏗️ Analogía de infraestructura.** Es un **`grep` que sabe ponderar**. El **IDF** es como una alerta: si salta en **todos** los hosts, no te dice nada; si salta en **uno**, es una señal fuerte. Los **rendimientos decrecientes** son como "10 alertas del mismo host no son 10 veces más graves que una". Y la **normalización** evita que un log gigante gane solo por tener más líneas.

**✏️ Ejemplo a mano.**

*TF con rendimientos decrecientes* (fragmento de largo promedio, `k1 = 1.5`): aparecer 1 vez aporta **1.00**; 2 veces, **1.43**; 3 veces, **1.67**; 10 veces, **2.17**. Diez apariciones no valen diez veces.

*IDF* (corpus de 10 fragmentos): "el" está en los 10 → **0.05**; "plan" en 6 → **0.53**; "opt-1107" en 1 → **1.99**. La palabra rara pesa unas 40 veces más que "el".

*Longitud:* con la misma palabra una vez, un fragmento del **doble** del largo medio aporta ≈ 0.69; uno de la **mitad**, ≈ 1.29.

**⚠️ Cuándo pierde.** No entiende significado: "contraseña" y "clave" son palabras distintas. Con una pregunta muy coloquial que no comparte palabras con el documento (paráfrasis), falla. Y con una palabra común del dominio ("casa") puede llevarte a una sección equivocada solo porque aparece.

**🐞 Errores típicos.** (1) Un tokenizador que **rompe los códigos** (separar `E-217` en `E` y `217` sin conservar el completo). (2) Aplicar *stemming* a los códigos. (3) No quitar palabras vacías. (4) Esperar que "entienda" sinónimos.

| Gana la… | Cuando la pregunta… |
| --- | --- |
| **semántica** | usa otras palabras que el documento (paráfrasis, intención) |
| **BM25** | contiene un código, número o nombre exacto |
| **híbrida** | puede ser de cualquiera de los dos tipos (lo normal en producción) |

Compruébalo con números en la celda siguiente (usa la clase `BM25` real con tres fragmentos de juguete):

In [ ]:
# Ejemplo a mano: los números de BM25 y un mini-corpus de tres fragmentos (sin LLM, sin embeddings, sin costo)
import math
from almacenes import Fragmento

k1, b = 1.5, 0.75
print("TF con rendimientos decrecientes (fragmento de largo promedio):")
for f in (1, 2, 3, 10):
    print(f"   la palabra aparece {f:>2} vez/veces → aporta {f * (k1 + 1) / (f + k1):.2f}")

n = 10
print("\nIDF con 10 fragmentos en el corpus:")
for palabra, d in (("el", 10), ("plan", 6), ("opt-1107", 1)):
    print(f"   {palabra!r:<11} está en {d:>2} de {n} fragmentos → IDF {math.log(1 + (n - d + 0.5) / (d + 0.5)):.2f}")

print("\nNormalización por longitud (la palabra aparece 1 vez):")
for etiqueta, rel in (("fragmento del DOBLE del largo medio", 2.0), ("fragmento de largo medio", 1.0), ("fragmento de la MITAD", 0.5)):
    print(f"   {etiqueta:<38} → aporta {1 * (k1 + 1) / (1 + k1 * (1 - b + b * rel)):.2f}")

mini = [Fragmento(id=f"F{i}", texto=t, metadatos={"fuente": "x", "seccion": "s"}) for i, t in enumerate([
    "La alarma OPT-1107 indica voltaje de alimentación alto en el equipo.",
    "La alarma OPT-1102 indica pérdida de señal óptica en el equipo.",
    "El plan Familiar incluye router y soporte técnico todos los días.",
])]
print("\nConsulta: '¿Qué indica la alarma OPT-1107?'")
for r in BM25(mini).buscar("¿Qué indica la alarma OPT-1107?", k=3):
    print(f"   {r.id}  {r.puntaje:.2f}  {r.texto[:60]}")
print("   (F1 también puntúa por 'alarma', 'indica' y la parte 'opt' del código, pero no por '1107'; F2 no comparte nada)")

## 5.1 Cómo se "ve" un texto: el tokenizador

In [ ]:
for texto in ["¿Qué significa el error E-217?", "Cancelar el contrato de Negocio", "Llamar al 0800-26337"]:
    print(f"{texto!r}\n   → {tokenizar(texto)}")
print("\nSin recortar palabras (prefijo=None):", tokenizar("Cancelación y cancelar", prefijo=None))
print("Con recorte a 5 letras (por defecto):  ", tokenizar("Cancelación y cancelar"))

Tres decisiones del tokenizador, y por qué:

- **Minúsculas y sin acentos:** "Dólares" = "dolares".
- **Códigos completos:** `e-217` se conserva **entero** (y también sus partes `e` y `217`), para que una búsqueda por el código exacto lo encuentre.
- **Recorte a 5 letras** (un *stemming* ingenuo): junta "cancelar", "cancelación" y "cancela" en una sola raíz. Es tosco, pero funciona sin librerías de lingüística. **No se aplica a tokens con dígitos**, para no estropear códigos.

## 5.2 Cuándo gana BM25 y cuándo pierde

In [ ]:
if HAY_EMB:
    for consulta in ["¿Qué indica la alarma OPT-1107?", "¿Pueden llevarme el servicio a otra casa?"]:
        print(f"━━ {consulta}")
        for nombre, funcion in (("semántica", buscar_vectorial), ("BM25", buscar_bm25)):
            r = funcion(indice_base, consulta, 1)[0]
            print(f"   {nombre:<10} → {r.metadatos['seccion'][:42]:<42} | {r.texto[:70]}…")
        print()

El primero es un **código** (BM25 lo encuentra, la semántica puede dudar); el segundo es una **paráfrasis** (la semántica entiende "otra casa" = "traslado"; BM25 no tiene ninguna palabra en común). Midámoslo por tipo de pregunta.

In [ ]:
def tabla_por_tipo(buscadores, tipos=("semantica", "lexica", "tabla"), k=3):
    """Una fila por buscador, una columna por tipo de pregunta (aciertos en el primer lugar) y el total."""
    print(f"{'buscador':<26}" + "".join(f"{t:>14}" for t in tipos) + f"{'TOTAL hit@1':>14}{'MRR':>7}")
    for nombre, fn in buscadores.items():
        celdas, total_aciertos, total = [], 0, 0
        for t in tipos:
            qs = [p for p in preguntas if p["tipo"] == t]
            r = evaluar_busqueda(qs, fn, k)
            celdas.append(f"{r['hit1']}/{r['n']}")
            total_aciertos += r["hit1"]; total += r["n"]
        todas = evaluar_busqueda([p for p in preguntas if p["tipo"] in tipos], fn, k)
        print(f"{nombre:<26}" + "".join(f"{c:>14}" for c in celdas) + f"{total_aciertos:>10}/{total:<3}{todas['mrr']:>7.2f}")

if HAY_EMB:
    tabla_por_tipo({
        "Semántica (vectores)": lambda p, k: buscar_vectorial(indice_base, p["pregunta"], k),
        "BM25 (palabras)": lambda p, k: buscar_bm25(indice_base, p["pregunta"], k),
    })

**Lectura:** BM25 gana en las preguntas **léxicas** (códigos, alarmas) y pierde en las **semánticas** (paráfrasis). Ninguna es mejor en general: cada una **cubre el punto ciego de la otra**. Eso nos lleva a combinarlas.

> 📊 **Referencia del instructor** (oraciones de 70 palabras): en las preguntas léxicas, semántica 8/11 y BM25 **11/11**; en las semánticas, semántica 8/9 y BM25 6/9.

---
# Parte 6 · Búsqueda híbrida y fusión RRF

## La idea

Hacemos **las dos búsquedas** y **juntamos** los resultados. El problema: ¿cómo se combinan dos listas con puntajes en escalas incomparables? (El coseno va de 0 a 1; BM25, de 0 a 20 o más.) Sumarlos no tiene sentido.

## RRF: *Reciprocal Rank Fusion*

La solución más usada es **ignorar los puntajes y usar solo la posición**:

$$\text{puntaje}(d) = \sum_{\text{listas}} \frac{\text{peso}}{c + \text{posición}(d)} \qquad (c = 60 \text{ por tradición})$$

Un fragmento que aparece **bien ubicado en ambas listas** suma dos aportes y sube al primer lugar. Un fragmento que solo aparece en una, queda atrás de uno que aparece en las dos. Es una **votación**.

### Búsqueda híbrida y RRF, aterrizadas

**🧠 Qué es.** La **búsqueda híbrida** ejecuta dos buscadores (semántico y BM25) y **combina** sus listas. **RRF** (*Reciprocal Rank Fusion*) es la forma de combinarlas usando **solo la posición** de cada fragmento, no su puntaje.

**⚙️ Qué hace.** Cada lista le da a cada fragmento `peso / (60 + posición)`; se suman los aportes de todas las listas y se reordena. Un fragmento bien ubicado en **ambas** listas suma dos aportes y sube.

**🎯 Para qué sirve.** Para tener **lo mejor de los dos**: encontrar tanto la paráfrasis como el código exacto, sin tener que decidir de antemano qué tipo de pregunta es.

**🏗️ Analogía de infraestructura.** Dos sistemas de monitoreo (métricas y logs) te dan **cada uno su ranking** de "servicios sospechosos", con **escalas distintas** (uno en porcentaje, otro en conteo de errores). Sumar esos números no tiene sentido. Lo robusto es mirar **en qué puesto** puso cada uno a cada servicio: el que aparece arriba en **los dos** es el primero que investigas. Es una **votación**.

**✏️ Ejemplo a mano (el de la lámina).** BM25 ordena **A, C, B**; la búsqueda por significado ordena **B, A, D**:

| Fragmento | Aporte BM25 | Aporte significado | Suma |
| --- | --- | --- | --- |
| **A** | 1/(60+1) = 0.01639 | 1/(60+2) = 0.01613 | **0.0325** |
| **B** | 1/(60+3) = 0.01587 | 1/(60+1) = 0.01639 | **0.0323** |
| **C** | 1/(60+2) = 0.01613 | (no aparece) | **0.0161** |
| **D** | (no aparece) | 1/(60+3) = 0.01587 | **0.0159** |

Resultado: **A, B, C, D**. Ni C ni D ganan: aparecen en una sola lista.

**¿Por qué la constante 60?** Suaviza las diferencias entre los primeros puestos. Con `c = 0`, el 1.º vale el doble que el 2.º (1 contra 0.5): un solo voto manda. Con `c = 60`, el 1.º y el 2.º valen casi lo mismo (0.0164 contra 0.0161): para ganar hay que estar bien en **varias** listas.

**¿Por qué no sumar los puntajes?** El coseno va de 0 a 1 y BM25 puede dar 12 o 20: si sumas, BM25 domina siempre y la semántica se vuelve decorado. Son escalas incomparables.

**⚠️ Cuándo no.**
- Con un corpus **pequeño**, la híbrida con pesos 1:1 puede quedar **entre** las dos (en nuestro corpus iguala a la semántica: 18/22).
- Los **pesos** importan, pero ajustarlos con las mismas preguntas con que evalúas es **hacer trampa** (sobreajuste): separa preguntas de **ajuste** y de **prueba**.
- La híbrida brilla cuando el **corpus crece** y la semántica empieza a confundir códigos parecidos.

**🐞 Errores típicos.** (1) Normalizar y sumar puntajes. (2) Fusionar listas **muy cortas** (pocos candidatos): trae cada buscador al menos 10. (3) Aplicar el filtro **después** de fusionar (hay que filtrar en cada búsqueda, antes).

### Un ejemplo a mano

In [ ]:
from almacenes import Resultado

def lista(*ids):
    return [Resultado(i, f"texto {i}", {}, 0.0) for i in ids]

semantica = lista("A", "B", "C")       # el orden según la búsqueda por significado
palabras  = lista("C", "A", "D")       # el orden según BM25
for r in fusion_rrf([semantica, palabras], k=4):
    print(f"{r.id}: {r.puntaje:.5f}")
print()
print("A: 1/(60+1) + 1/(60+2) =", round(1 / 61 + 1 / 62, 5), " (primero en una lista, segundo en la otra)")
print("C: 1/(60+3) + 1/(60+1) =", round(1 / 63 + 1 / 61, 5), " (tercero en una, primero en la otra)")
print("B: 1/(60+2)            =", round(1 / 62, 5), " (solo aparece en una lista)")

#### El mismo cálculo con el ejemplo de la lámina

El código de arriba usa **otras dos listas** (semántica A, B, C y BM25 C, A, D); la lámina de clase usa BM25 **A, C, B** y significado **B, A, D**. El método es idéntico. Verifica la tabla que acabas de ver:

In [ ]:
# Mismo RRF, con las listas de la lámina: BM25 → A, C, B · significado → B, A, D
bm25_demo = lista("A", "C", "B")
significado_demo = lista("B", "A", "D")
for r in fusion_rrf([bm25_demo, significado_demo], k=4):
    print(f"{r.id}: {r.puntaje:.4f}")
print("\nA: 1/61 + 1/62 =", round(1 / 61 + 1 / 62, 4), "· B: 1/63 + 1/61 =", round(1 / 63 + 1 / 61, 4),
      "· C: 1/62 =", round(1 / 62, 4), "· D: 1/63 =", round(1 / 63, 4))

## 6.1 ¿Funciona? Comparemos cuatro buscadores

Añadimos una variante: **pesos distintos** (`[1, 2]` = BM25 vale el doble que la semántica).

In [ ]:
if HAY_EMB:
    tabla_por_tipo({
        "Semántica": lambda p, k: buscar_vectorial(indice_base, p["pregunta"], k),
        "BM25": lambda p, k: buscar_bm25(indice_base, p["pregunta"], k),
        "Híbrida (pesos 1 : 1)": lambda p, k: buscar_hibrido(indice_base, p["pregunta"], k),
        "Híbrida (semántica 1 : BM25 2)": lambda p, k: buscar_hibrido(indice_base, p["pregunta"], k, pesos=[1, 2]),
    })

**Lo que probablemente verás, y por qué es una lección honesta:**

- La híbrida con pesos **1 : 1 no siempre supera a la mejor de las dos**: en este corpus pequeño la semántica ya es muy buena, y BM25 "estorba" en las paráfrasis, así que el híbrido iguala o queda **entre las dos**.
- Con pesos que **favorecen a BM25** mejora porque rescata los códigos sin perder demasiado en las paráfrasis. Aquí aparece un riesgo: **elegir los pesos con las mismas preguntas con las que evalúas es hacer trampa** (sobreajuste). En serio, separas preguntas de **ajuste** y de **prueba**.
- La híbrida **gana de verdad cuando el corpus crece**: con miles de fragmentos, la semántica empieza a confundir códigos parecidos, y BM25 es el seguro. También es la forma **estándar** en bases como Elasticsearch, OpenSearch, Pinecone (híbrido), Weaviate o PostgreSQL (`tsvector` + pgvector).

> 📊 **Referencia del instructor** (oraciones de 70 palabras, 22 preguntas): semántica 18/22 y MRR 0.87; BM25 19/22 y 0.86; híbrida 1 : 1, 18/22 y 0.86; híbrida con BM25 doble, **20/22 y 0.91**.

> 🏗️ **Analogía:** la semántica es el **compañero senior que entiende lo que quieres decir**; BM25 es el **`grep`**. Quieres a los dos en la guardia, y alguien que decida cuando discrepan (RRF).

---
# Parte 7 · Metadatos y filtros

## Dos preguntas, el mismo texto, respuestas distintas

> **"¿En cuánto tiempo deben reparar una falla?"**
> - A un cliente **residencial**: 48 horas. 
> - A un cliente **Negocio**: 4 horas.

Un buscador sin contexto devuelve los dos fragmentos (se parecen muchísimo) y el LLM puede **mezclar las respuestas**. La solución es **filtrar por metadatos**: no es una cuestión de "significado", es una **regla de negocio**: *"este usuario es un cliente Negocio"*.

El etiquetado ocurre en la **ingesta** (`publico` se infiere del título de la sección del DOCX), y el filtro se aplica en la **búsqueda**, **antes** de que nada llegue al prompt.

### Metadatos y filtros, aterrizados

**🧠 Qué es.** Los **metadatos** son datos **sobre** cada fragmento (de qué archivo viene, de qué sección, de qué tipo, **para qué audiencia**, de qué fecha). Un **filtro** es una restricción sobre esos metadatos que se aplica **dentro** de la búsqueda: "solo candidatos con `publico = negocio`".

**⚙️ Qué hace.** Se **asignan en la ingesta** (por ejemplo, `publico` sale del título de la sección del Word) y se **usan en la consulta** para descartar candidatos *antes* de ordenar.

**🎯 Para qué sirve.** Para cuatro cosas distintas:

| Uso | Ejemplo |
| --- | --- |
| **Calidad** | Dos fragmentos casi idénticos, respuestas distintas según el tipo de cliente |
| **Seguridad** | Cada usuario o inquilino ve **solo** sus documentos |
| **Frescura** | Quedarse con la versión vigente de un documento |
| **Ámbito** | Buscar solo en un manual, producto o idioma |

**🏗️ Analogía de infraestructura.** Son las **etiquetas (*labels*) de Kubernetes o Prometheus** (`env=prod`, `team=pagos`) y el **control de acceso (RBAC / seguridad por fila)** de una base de datos. Filtrar en la búsqueda es poner la **ACL en la base**; "filtrar en el prompt" sería pedirle al usuario que **prometa** no mirar las filas ajenas.

**✏️ Ejemplo a mano.** Hay dos fragmentos casi iguales: "reparamos en un máximo de **48 horas**" (`publico = residencial`) y "reparamos en un máximo de **4 horas**" (`publico = negocio`). Una consulta de un cliente Negocio **sin filtro** devuelve ambos y el LLM puede mezclarlos. **Con** `publico = negocio`, el de 48 horas ni se considera.

**Filtrar antes, no después.** Si buscas los 3 mejores y *luego* filtras, puedes quedarte con **cero** resultados (los 3 mejores eran de otra audiencia). Filtrar **dentro** de la búsqueda devuelve los 3 mejores *que cumplen* la regla.

**⚠️ Cuándo no basta.**
- El filtro decide **qué se puede ver**, no **qué es más relevante**: sigue haciendo falta buscar bien.
- Si el metadato está mal asignado en la ingesta, el filtro **esconde** documentos correctos (o muestra los equivocados).

**🐞 Errores típicos.** (1) **Filtrar en el prompt** ("no muestres lo de otros clientes"): el dato ya llegó al modelo. (2) Dejar que el **LLM o el texto de la pregunta** decidan el filtro: el filtro lo decide **tu código** a partir de la sesión del usuario. (3) No **auditar** los metadatos con que se filtra. (4) Olvidar un valor por defecto (`general`) para lo que no tiene audiencia.

In [ ]:
if HAY_EMB:
    from collections import Counter
    print("Fragmentos por audiencia:", dict(Counter(f.metadatos["publico"] for f in fragmentos_base)))
    print("Fragmentos por tipo de archivo:", dict(Counter(f.metadatos["tipo"] for f in fragmentos_base)))

    pregunta = "¿En cuánto tiempo deben reparar una falla?"
    for etiqueta, filtro in (("SIN filtro", None), ("con publico=negocio", {"publico": "negocio"}), ("con publico=residencial", {"publico": "residencial"})):
        print(f"\n━━ {etiqueta}")
        for r in buscar_hibrido(indice_base, pregunta, 2, filtro):
            print(f"   [{r.metadatos['publico']:<11}] {r.metadatos['seccion'][:50]:<50} | {r.texto[:55]}…")

In [ ]:
if HAY_EMB:
    filtradas = [p for p in preguntas if p["tipo"] == "filtro"]
    print(f"{len(filtradas)} preguntas cuya respuesta depende del tipo de cliente (cada una trae su filtro)\n")
    for etiqueta, usar in (("SIN filtro", False), ("CON filtro", True)):
        for nombre, funcion in (("semántica", buscar_vectorial), ("híbrida", buscar_hibrido)):
            r = evaluar_busqueda(filtradas, lambda p, k: funcion(indice_base, p["pregunta"], k, p["filtro"] if usar else None), 3)
            print(f"{etiqueta:<11} {nombre:<10} hit@1 {r['hit1']}/{r['n']} · hit@3 {r['hitk']}/{r['n']} · MRR {r['mrr']:.2f}")

## Los filtros no son solo para mejorar la calidad: son **seguridad**

Si tu base mezcla documentos de **varios clientes, equipos o niveles de confidencialidad**, el filtro por permisos (`tenant`, `area`, `nivel`) es la **única** barrera real. Reglas de oro:

1. **Filtra en la búsqueda, no en el prompt.** Un prompt que diga "no muestres lo de otros clientes" **no es un control de acceso**: el dato ya llegó al modelo.
2. El filtro lo decide **tu código** (a partir de la sesión del usuario), **nunca el modelo** ni el texto de la pregunta.
3. Los metadatos que usas para filtrar se **asignan en la ingesta** y se **auditan**.
4. Las bases reales (Neon con `WHERE`, Pinecone con `filter`) aplican el filtro **dentro** de la búsqueda; el almacén en memoria de esta sesión hace lo mismo.

> En la Sesión 5 lo mencionamos como riesgo (*fuga de permisos*). Aquí ya sabes cómo se evita.

---
# Parte 8 · Reranking

## Dos etapas: primero amplio, luego fino

Los embeddings comprimen **cada texto por separado** en un vector, y luego se comparan. Es rapidísimo (miles de fragmentos en milisegundos), pero **pierde detalle**: nunca "lee" la pregunta y el fragmento **a la vez**.

El **reranking** añade una segunda etapa:

```
Etapa 1 · RECUPERAR (rápida, amplia)     Etapa 2 · REORDENAR (lenta, fina)
  búsqueda híbrida → 8 candidatos   ──►    un modelo lee pregunta + cada candidato  ──►  los 3 mejores al prompt
```

| | Embeddings (*bi-encoder*) | Reranker (*cross-encoder*) |
| --- | --- | --- |
| Cómo lee | Cada texto **por separado** | Pregunta **y** fragmento **juntos** |
| Velocidad | Muy rápido (se precalcula) | Lento (una pasada por cada candidato) |
| Precisión | Buena | **Mejor** |
| Se usa para | Buscar entre miles | **Reordenar** unas decenas |

🏗️ **Analogía:** la etapa 1 es el **filtro de currículos** que reduce 2 000 candidatos a 8; la etapa 2 es la **entrevista** (cara, lenta, pero solo con 8).

### Qué se usa en producción

Un **cross-encoder especializado** (modelos como `bge-reranker`, o APIs de Cohere, Voyage o Jina). Son más rápidos, baratos y fiables que un LLM general. En esta sesión usamos un **LLM como reranker** porque no exige instalar nada: es el mismo cliente de siempre.

Lo implementamos con **dos versiones de prompt** (versionadas en `prompts/rerank/`, como en la Sesión 3):

- **v1:** "ordena **todos** los fragmentos de más a menos útil".
- **v2:** "elige **solo el mejor**" (una tarea mucho más fácil para un modelo pequeño).

💰 **FYI:** cada pregunta es **1 llamada** con 8 candidatos (≈ 1 300 tokens de entrada). 22 preguntas × 2 versiones ≈ 44 llamadas: **≈ US$ 0.006** con un modelo económico; gratis con `RAG_PROVIDER=local` (tarda 1 o 2 minutos).

### Reranking, aterrizado

**🧠 Qué es.** Una **segunda etapa** que **reordena** los candidatos que trajo la primera. La primera etapa es rápida y amplia; la segunda es lenta y fina, y solo ve unas pocas decenas de candidatos.

**⚙️ Qué hace.** Un modelo lee la **pregunta y el fragmento juntos** y le pone una nota de relevancia. Los embeddings (**bi-encoder**) leen cada texto **por separado** y comparan vectores; un **cross-encoder** (el reranker) compara **con ambos textos en la mano**, así que ve matices que el vector perdió.

**🎯 Para qué sirve.** Para rescatar el fragmento correcto cuando quedó en el puesto 5 pero solo enviamos 3 al LLM. Pasas de "recall" (que esté entre los candidatos) a "precisión" (que esté arriba).

**🏗️ Analogía de infraestructura.** Una **ACL/filtro de red** que decide en microsegundos por IP y puerto (barato, amplio) seguida de un **IDS con inspección profunda** que lee el contenido del paquete (caro, preciso) **solo** sobre lo que pasó el primer filtro. Nadie corre inspección profunda sobre todo el tráfico.

**✏️ Ejemplo a mano.** Etapa 1 trae 5 candidatos en este orden: **C3, C1, C5, C2, C4**. El fragmento correcto es **C2**, en el puesto 4. Si solo enviamos 3 al LLM (**C3, C1, C5**), **no está** y el LLM no puede responder. Con reranking, el modelo lee la pregunta junto a cada candidato, pone a C2 primero y enviamos **C2, C3, C1**. Eso mejora hit@3 y MRR. Pero ojo: si el correcto estuviera en el puesto 9 de una lista de 5 candidatos, **no existiría** para el reranker.

**⚠️ Por qué un modelo pequeño puede empeorar.** Ordenar una lista completa es una tarea difícil: el modelo tiene que comparar todo contra todo y devolver un formato exacto. Un modelo de ~3B parámetros se equivoca y **desordena** algo que ya estaba bien (en nuestras mediciones, ordenar todo bajó de 18/22 a 15/22). **Elegir solo el mejor** es una tarea más simple y rinde mejor. Un cross-encoder especializado (o un modelo grande) suele ser claramente mejor, pero **se mide con tus datos**.

**🐞 Errores típicos.** (1) Dar pocos candidatos a la etapa 1 (si el correcto no entra, no se rescata). (2) Reordenar con un LLM pequeño sin medir. (3) Ignorar la latencia: es una llamada extra **en el camino de la respuesta**. (4) Mandar fragmentos enormes al reranker (se recortan).

**Cuándo no:** si tu hit@3 ya es alto y la etapa 1 deja lo correcto arriba, el reranking solo añade costo.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.006 con un modelo de proveedor; gratis con RAG_PROVIDER=local)
if PUEDE_GASTAR:
    CANDIDATOS = 8

    def con_rerank(version):
        def buscador(p, k):
            candidatos = buscar_hibrido(indice_base, p["pregunta"], CANDIDATOS)
            resultados, llamada = rerank_llm(p["pregunta"], candidatos, k=k, version=version)
            if llamada:
                gasto.registrar(llamada)
            return resultados
        return buscador

    t0 = time.perf_counter()
    linea("Híbrida, sin rerank (top 3)", evaluar_busqueda(BASE, lambda p, k: buscar_hibrido(indice_base, p["pregunta"], k), 3))
    linea("Híbrida + rerank v1 (ordenar todos)", evaluar_busqueda(BASE, con_rerank("v1"), 3))
    linea("Híbrida + rerank v2 (elegir el mejor)", evaluar_busqueda(BASE, con_rerank("v2"), 3))
    print(f"\n({time.perf_counter() - t0:.0f} s)")
else:
    print("⚪ Celda de pago desactivada o sin LLM disponible.")

## Cómo interpretar esto

- **Un reranker no es magia.** Reordena **solo lo que la etapa 1 trajo**: si el fragmento correcto no está entre los 8 candidatos, nadie lo rescata. Por eso se sube `CANDIDATOS` en la etapa 1 (8 a 50) y se baja `k` en la etapa 2 (3 a 5).
- **Con un modelo pequeño puede empeorar.** Un LLM de 3B parámetros ordena una lista completa **peor** que el orden original; elegir **solo el mejor** (v2) es más fiable, aunque aquí apenas iguala el orden original. Con un modelo de proveedor, o con un cross-encoder, el resultado suele ser claramente mejor. **Tienes que medirlo con tus datos.**
- **Cuesta latencia y dinero**: una llamada extra por pregunta, en el camino crítico de la respuesta.

> 📊 **Referencia del instructor** (modelo local `llama-3.2-3b-instruct`, híbrida con 8 candidatos, 22 preguntas): sin rerank, 18/22 en hit@1 y MRR 0.86; rerank v1, **15/22** y 0.80 (**empeora**); rerank v2, 16 a 17/22 en hit@1 (según la corrida), hit@3 de 20 a **22/22** y MRR 0.86 a 0.89 (no mejora el primer lugar; rescata algún fragmento en el top 3). Los modelos no son del todo deterministas: un resultado puede variar 1 o 2 aciertos entre corridas. No se probó con un modelo de proveedor.

**Reto:** vuelve a ejecutar con `RAG_PROVIDER=anthropic` u `openai` (si tienes llave) y compara. ¿Cambia la conclusión?

---
# Parte 9 · Reescritura de la consulta y multi-consulta

El usuario **no escribe consultas de búsqueda**: escribe como habla. Dos problemas, dos técnicas.

### Las técnicas sobre la consulta, aterrizadas

Las tres comparten la idea de **mejorar la pregunta antes de buscar**, pero atacan problemas distintos:

| | **Reescritura** (*query rewriting*) | **Multi-consulta** (*multi-query*) | **HyDE** |
| --- | --- | --- | --- |
| 🧠 **Qué es** | Convertir la pregunta en una consulta **autocontenida** usando el historial | Generar **varias formas de preguntar** lo mismo | Pedir al LLM una **respuesta hipotética** y buscar con *esa* |
| ⚙️ **Qué hace** | "¿Y los domingos?" → "¿A qué hora atienden por teléfono los domingos?" | Busca con la original y N variantes y **fusiona con RRF** | Vectoriza la respuesta inventada (se parece más a un documento que una pregunta) |
| 🎯 **Para qué sirve** | Preguntas de seguimiento, pronombres, referencias | El usuario usa **otras palabras** ("multa" vs "penalización") | Preguntas **cortas o vagas** |
| 🏗️ **Analogía** | Un ticket que dice "sigue igual": el ingeniero de turno lo **reescribe con el historial** antes de buscar en la base de conocimiento | Buscar en la wiki con sinónimos: "caída", "outage", "incidente" | Buscar un runbook escribiendo primero **cómo imaginas que sería su solución** |
| ⚠️ **Riesgo** | Perder o inventar un dato del historial | Las variantes **inventan temas** y arrastran fragmentos ajenos | Si el modelo inventa, la búsqueda se desvía |
| 🐞 **Error típico** | Reescribir cuando **no hay historial** | Usar un modelo pequeño sin medir | Tratarla como mejora segura |

**✏️ Ejemplo a mano (reescritura).** Historial: *Cliente: "¿A qué hora atienden por teléfono de lunes a sábado?" · Asistente: "De 7:00 a 22:00."* Nueva pregunta: *"¿Y los domingos?"*. Sin reescribir, el buscador recibe tres palabras sin tema y trae cualquier sección que hable de domingos. Reescrita, recibe la pregunta completa y encuentra el horario.

**Qué cuesta cada una.** Reescritura: 1 llamada pequeña por pregunta (solo con historial). Multi-consulta: 1 llamada para generar las variantes y N+1 búsquedas. HyDE: 1 llamada para generar el texto hipotético. Todas añaden **latencia y un punto de falla**.

**Regla de oro:** reescribe **solo con historial**, **registra la consulta reescrita** (es lo primero que se revisa cuando algo falla) y pide al modelo que **conserve códigos y nombres** y que **no responda**, solo reformule. En nuestras mediciones con un modelo pequeño, la reescritura ayudó un poco y la multi-consulta **empeoró** (8/9 → 5/9).

## 9.1 Reescritura: preguntas que no se entienden solas

> **Cliente:** ¿A qué hora atienden por teléfono de lunes a sábado?
> **Asistente:** De 7:00 a 22:00.
> **Cliente:** **¿Y los domingos?**

Si buscas con "¿Y los domingos?", el buscador no sabe de qué hablas. La solución: pedirle a un LLM que **reescriba la última pregunta como una consulta autocontenida** usando el historial: *"¿A qué hora atienden por teléfono los domingos?"*. La plantilla vive en `prompts/reescritura/v1.j2`.

💰 **FYI:** 4 llamadas pequeñas (≈ 300 tokens cada una): **≈ US$ 0.0005**; gratis con `local`.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0005 con un modelo de proveedor; gratis con RAG_PROVIDER=local)
conversacionales = [p for p in preguntas if p["tipo"] == "conversacional"]
reescritas = {}
if PUEDE_GASTAR:
    for p in conversacionales:
        consulta, llamada = reescribir_consulta(p["pregunta"], p["historial"])
        gasto.registrar(llamada)
        reescritas[p["id"]] = consulta
        print(f"{p['id']}  {p['pregunta']!r}\n      → {consulta!r}\n")
else:
    print("⚪ Celda de pago desactivada o sin LLM disponible.")

In [ ]:
if PUEDE_GASTAR and reescritas:
    for nombre, funcion in (("semántica", buscar_vectorial), ("híbrida", buscar_hibrido)):
        sin = evaluar_busqueda(conversacionales, lambda p, k: funcion(indice_base, p["pregunta"], k, p.get("filtro")), 3)
        con = evaluar_busqueda(conversacionales, lambda p, k: funcion(indice_base, reescritas[p["id"]], k, p.get("filtro")), 3)
        print(f"{nombre:<10} SIN reescribir: hit@1 {sin['hit1']}/{sin['n']} · MRR {sin['mrr']:.2f}   |   "
              f"CON reescritura: hit@1 {con['hit1']}/{con['n']} · MRR {con['mrr']:.2f}")

**Lectura:** reescribir ayuda con las preguntas de seguimiento, pero **no es perfecta**: con un modelo pequeño verás consultas que **pierden un dato** del historial (por ejemplo, el nombre del plan) o **lo inventan**. Con solo 4 preguntas es una **ilustración**, no una prueba. Dos reglas prácticas: (1) reescribe **solo cuando hay historial**; (2) **muestra la consulta reescrita** en tus registros: cuando el RAG falle, es lo primero que querrás mirar.

> 📊 **Referencia del instructor** (`llama-3.2-3b-instruct`, 4 preguntas): híbrida, hit@1 de 1/4 a 2/4 y MRR de 0.50 a ≈ 0.60; semántica, de 0/4 a 1/4. En una de las cuatro, el modelo perdió el nombre del plan ("¿Y trae IP fija?" quedó como "¿Trata de IP fija?").

## 9.2 Multi-consulta: preguntar de varias maneras

Otra idea: pedir al LLM **N reformulaciones** de la pregunta, buscar con **todas** y **fusionar** con RRF. Ayuda cuando el usuario usa palabras distintas a las del documento ("multa" vs "penalización").

💰 **FYI:** 9 llamadas pequeñas (una por pregunta semántica): **≈ US$ 0.001**; gratis con `local`.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.001 con un modelo de proveedor; gratis con RAG_PROVIDER=local)
semanticas = [p for p in preguntas if p["tipo"] == "semantica"]
if PUEDE_GASTAR:
    variantes = {}
    for p in semanticas:
        v, llamada = expandir_consulta(p["pregunta"], n=3)
        gasto.registrar(llamada)
        variantes[p["id"]] = v
    print("Ejemplo:", semanticas[1]["pregunta"])
    for v in variantes[semanticas[1]["id"]]:
        print("   ·", v)
    print()
    linea("Semántica (una consulta)", evaluar_busqueda(semanticas, lambda p, k: buscar_vectorial(indice_base, p["pregunta"], k), 3))
    linea("Multi-consulta (original + 3)",
          evaluar_busqueda(semanticas, lambda p, k: buscar_multiconsulta(indice_base, [p["pregunta"]] + variantes[p["id"]], k, modo="vectorial"), 3))
else:
    print("⚪ Celda de pago desactivada o sin LLM disponible.")

**Lectura:** con un modelo pequeño, la multi-consulta **empeora**: las "reformulaciones" **inventan temas** (otro plan, otra garantía) y arrastran fragmentos que no corresponden. Con un modelo fuerte suele ayudar. **Moraleja de las partes 8 y 9: cada paso con LLM añade latencia, dinero y un punto de falla; se queda solo si la medición lo justifica.**

> 📊 **Referencia del instructor** (`llama-3.2-3b-instruct`, 9 preguntas semánticas): una consulta, 8/9 en hit@1 y MRR 0.93; multi-consulta, **5/9** y 0.70.

### HyDE (solo concepto)

*Hypothetical Document Embeddings*: en lugar de vectorizar la pregunta, se le pide al LLM que **escriba una respuesta hipotética** y se vectoriza **esa respuesta** (que se parece más a un documento que una pregunta). Funciona a veces; tiene el mismo problema: si el modelo inventa, la búsqueda se desvía.

---
# Parte 10 · El pipeline completo y la práctica

## Todas las piezas juntas

`responder_avanzado` conecta lo que viste en esta sesión con la generación con citas de la Sesión 5:

```
pregunta ─► [reescribir si hay historial] ─► búsqueda híbrida + FILTRO ─► [rerank] ─► prompt con citas ─► LLM ─► verificar citas
```

Los pasos entre corchetes son **opcionales**: activa solo los que **mediste que ayudan**. Devuelve la respuesta y un diccionario con lo que pasó en cada paso (para **depurar**).

💰 **FYI:** 3 preguntas; con todos los pasos, ≈ 3 llamadas cada una: **≈ US$ 0.003**; gratis con `local`.

### Cómo decidir qué piezas activar

No todo RAG necesita todas las piezas. Parte de la **línea base** y añade una pieza **solo si el síntoma lo pide** y la medición lo confirma:

| Síntoma que ves | Primera pieza a probar | Cómo confirmar |
| --- | --- | --- |
| Faltan documentos o salen con basura | Ingesta / OCR | Muestra del texto extraído; páginas sin texto |
| Respuestas cortadas o ruido en el contexto | Chunking (o padre-hijo) | Tabla del laboratorio: hit@k **y contexto** |
| No encuentra códigos, números o nombres | BM25 / híbrida | Tabla por tipo de pregunta (`lexica`) |
| Mezcla respuestas de clientes o versiones | Metadatos y filtros | Preguntas de tipo `filtro` con y sin filtro |
| El correcto aparece, pero lejos del primer puesto | Reranking | MRR y hit@3 antes y después |
| Falla en preguntas de seguimiento o vagas | Reescritura / multi-consulta | Preguntas `conversacional` con y sin reescribir |

**Orden recomendado de inversión** (de lo más barato y seguro a lo más caro): ingesta → chunking → filtros → híbrida → reranking → reescritura/multi-consulta/HyDE. Las tres últimas añaden un LLM por pregunta.

> **Para estudiar una implementación formal** (con interfaz web, citas, fragmentos recuperados y evaluación) de este mismo pipeline, ve `rag-playground/rag-intermedio` (repositorio `/home/oscar/Desktop/Personal/EDUTEK/GEN-AI/rag-playground`).

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.003 con un modelo de proveedor; gratis con RAG_PROVIDER=local)
casos = [
    dict(pregunta="¿Qué indica la alarma OPT-1107?"),
    dict(pregunta="¿Y los domingos?", historial=conversacionales[0]["historial"], reescribir=True),
    dict(pregunta="¿Cuánto dura el contrato?", filtro={"publico": "negocio"}),
]
if PUEDE_GASTAR:
    for caso in casos:
        respuesta, pasos = responder_avanzado(indice_base, **caso)
        for nombre, llamada in pasos["llamadas"]:
            gasto.registrar(llamada)
        print(f"━━ {caso['pregunta']}")
        if pasos["consulta"] != caso["pregunta"]:
            print(f"   consulta reescrita → {pasos['consulta']}")
        print(f"   respuesta: {respuesta.texto}")
        print(f"   recuperó:  {[f.metadatos['fuente'] + ' · ' + f.metadatos['seccion'] for f in respuesta.fragmentos]}")
        print(f"   citó:      {sorted(respuesta.citas) or 'nada (el modelo no citó ningún fragmento)'} · citas inválidas: {sorted(respuesta.citas_invalidas) or 'ninguna'}\n")
else:
    print("⚪ Celda de pago desactivada o sin LLM disponible.")

Fíjate en dos cosas: la **consulta reescrita** aparece en el diccionario de pasos (así se depura), y el modelo pequeño puede **responder bien y no citar**: en la Sesión 5 viste que citar es difícil para un modelo de 3B. Las respuestas sin cita **no son verificables**; con un modelo de proveedor suelen citar mejor.

## La práctica: Laboratorio de chunking

La práctica de la sesión es [`practica/laboratorio_chunking.py`](practica/laboratorio_chunking.py): un programa único y comentado que **prueba combinaciones de estrategia × tamaño × tipo de búsqueda** con tus documentos y tus preguntas, y te dice **cuál recupera mejor y a qué costo de contexto**. No llama al LLM: es gratis.

```bash
# Desde la carpeta s06-rag-intermedio/
uv run python practica/laboratorio_chunking.py
uv run python practica/laboratorio_chunking.py --modo hibrido --detalle
uv run python practica/laboratorio_chunking.py --estrategias padre-hijo,seccion,oraciones --tamanos 35,70
uv run python practica/laboratorio_chunking.py --tipos lexica
uv run python practica/laboratorio_chunking.py --estrategias oraciones,docling-hibrido --tokens 64,128,256   # requiere Docling
uv run python practica/laboratorio_chunking.py --documentos mi_carpeta/ --preguntas mis_preguntas.json
```

### Retos
1. **Tu propio corpus:** añade un documento a `datos/` (un reglamento, un manual, una página web guardada) y escribe **8 preguntas** con su `clave` en un JSON nuevo. ¿Gana la misma estrategia?
2. **El tamaño ideal:** prueba `--tamanos 20,40,70,120,200`. ¿Dónde empieza a empeorar? ¿Por qué?
3. **Una pregunta difícil:** escribe una pregunta que **ninguna** configuración responda bien y explica por qué (¿tabla aplanada? ¿término que el documento no usa?).
4. **Prompt de reescritura v2:** crea `prompts/reescritura/v2.j2`, anótalo en `prompts/CHANGELOG.md` y compáralo con v1 en las cuatro preguntas conversacionales.
5. **Detectar escaneos:** escribe una función que recorra una carpeta y liste los PDFs con páginas sin texto.
6. **Docling con tu documento:** pasa un PDF tuyo con una tabla por `leer_pdf_adaptativo` (en `parsing_docling.py`) y compara la tabla con la de `pypdf`. ¿Qué fragmentos cambian en el laboratorio con `docling-hibrido`?

---
# Parte 11 · Cierre

## Lo que aprendiste
1. La **ingesta** decide la calidad de todo lo demás: extrae texto **limpio y estructurado**; un escaneo sin OCR **no existe** para tu RAG.
2. El **chunking** equilibra precisión al buscar y contexto al responder; se elige **midiendo**, mirando también el **costo del contexto**.
3. **Padre-hijo:** buscar con fragmentos pequeños y entregar el grande.
4. La búsqueda **semántica** entiende paráfrasis; **BM25** no se confunde con códigos. La **híbrida con RRF** combina ambas, y los **pesos** importan.
5. Los **metadatos y filtros** resuelven reglas de negocio y de **seguridad** antes de que los datos lleguen al modelo.
6. El **reranking** y la **reescritura de la consulta** pueden ayudar, pero cada uno añade un LLM: **se justifican con mediciones**, no con moda.
7. Con pocas preguntas, las diferencias pequeñas son **ruido**.
8. Un **parser con análisis de layout** (Docling) reconstruye tablas y orden de lectura, y trae **OCR**; pero el OCR **se equivoca** (un apellido, un código) y hay que revisarlo. Es opcional: se usa cuando se mide que hace falta.
9. El **chunking híbrido de Docling** (jerarquía + tokens) decide **dónde cortar**; **no** es la búsqueda híbrida (BM25 + vectores), que decide **cómo buscar**. En nuestros datos empató con "oraciones": su valor está en las citas, las tablas y el tamaño en tokens.

## ✅ Autoevaluación
1. ¿Qué dos presiones opuestas hay al elegir el tamaño de un fragmento?
2. ¿Por qué la estrategia `seccion` "gana" en MRR y por qué eso puede ser engañoso?
3. Da un ejemplo de pregunta donde BM25 gane a la semántica y otro donde pierda.
4. ¿Por qué RRF usa la **posición** y no el puntaje?
5. ¿Por qué filtrar por permisos en la búsqueda y no en el prompt?
6. ¿Qué diferencia hay entre un *bi-encoder* y un *cross-encoder*? ¿Cuál usarías para reordenar 8 candidatos?
7. ¿Cuándo **no** reescribirías la consulta?
8. ¿Qué es el OCR y cómo te darías cuenta de que a tu RAG le faltan documentos escaneados?
9. Un PDF "no tiene tablas ni párrafos": ¿qué tiene, y qué añade el análisis de *layout*?
10. ¿Qué diferencia hay entre el **chunking híbrido** de Docling y la **búsqueda híbrida**? ¿Por qué el tokenizador del chunker debe coincidir con el del modelo de embeddings?

## 💰 Resumen de gasto de esta sesión

### Respuestas orientativas a la autoevaluación (intenta primero sin mirar)

1. **Dos presiones:** pequeño = vector preciso pero poco contexto; grande = contexto completo pero vector diluido, más tokens y más ruido.
2. **`seccion` gana en MRR** porque la sección entera casi siempre *contiene* la clave; eso engaña porque entrega el doble de contexto (≈ 345 contra ≈ 160 palabras).
3. **BM25 gana** con un código exacto (`OPT-1107`); **pierde** con una paráfrasis sin palabras en común ("llevarme el servicio a otra casa" = traslado).
4. **RRF usa la posición** porque los puntajes (coseno 0 a 1, BM25 0 a 20) son de escalas incomparables.
5. **Filtrar en la búsqueda**: en el prompt el dato ya llegó al modelo y no es un control de acceso.
6. **Bi-encoder** lee cada texto por separado (rápido, para buscar); **cross-encoder** lee pregunta y fragmento juntos (preciso, lento). Para reordenar 8 candidatos: el cross-encoder.
7. **No reescribir** cuando no hay historial: añade un LLM, latencia y riesgo de perder datos sin necesidad.
8. **OCR** convierte imagen en texto; te das cuenta contando las páginas sin texto en cada ingesta.
9. Tiene **letras en posiciones** (no filas ni párrafos). El análisis de layout detecta las **regiones** (título, párrafo, tabla…), reconstruye el **orden de lectura** y las **tablas**, y decide dónde hace falta OCR.
10. El **chunking híbrido** decide **dónde cortar** (jerarquía + tokens, en la ingesta); la **búsqueda híbrida** decide **cómo buscar** (BM25 + vectores, en la consulta). El tokenizador debe ser el del modelo de embeddings porque los tokens se cuentan distinto en cada modelo: si no, el límite engaña.

In [ ]:
gasto.resumen()

## 📖 Glosario

| Término | Definición |
| --- | --- |
| **Ingesta** | Extraer el texto de los documentos, fragmentarlo, vectorizarlo y guardarlo |
| **OCR** | Convertir una imagen de texto (un escaneo) en texto |
| **Chunking** | Dividir un documento en fragmentos para buscar y para dar contexto |
| **Solape (*overlap*)** | Repetir el final de un fragmento al inicio del siguiente |
| **Fragmentado recursivo** | Cortar por el separador más grande que quepa (párrafo, línea, oración, palabra) |
| **Fragmentado semántico** | Cortar donde cambia el tema según la similitud entre oraciones |
| **Padre-hijo** | Buscar con fragmentos pequeños (hijos) y entregar el fragmento grande (padre) |
| **BM25** | Algoritmo clásico de búsqueda por palabras (IDF + frecuencia + longitud) |
| **IDF** | Qué tan rara es una palabra en el corpus: las raras pesan más |
| **Búsqueda híbrida** | Combinar búsqueda por palabras y búsqueda semántica |
| **RRF** | *Reciprocal Rank Fusion*: combina listas ordenadas usando solo la posición |
| **Metadatos** | Datos sobre cada fragmento (fuente, sección, audiencia) |
| **Filtro** | Restricción por metadatos aplicada **dentro** de la búsqueda |
| **Reranking** | Reordenar los candidatos con un modelo que lee pregunta y fragmento juntos |
| **Cross-encoder** | Modelo que puntúa pregunta y fragmento **juntos**; más preciso y lento que un bi-encoder |
| **Reescritura de consulta** | Convertir una pregunta de seguimiento en una consulta autocontenida |
| **Multi-consulta** | Buscar con varias reformulaciones y fusionar los resultados |
| **HyDE** | Vectorizar una respuesta hipotética en lugar de la pregunta |
| **hit@k** | La respuesta está entre los k primeros resultados |
| **MRR** | Promedio de 1/posición del primer acierto |
| **Layout (análisis de)** | Reconocer las regiones de una página (título, párrafo, tabla, figura) y su orden de lectura |
| **Docling** | Convertidor de documentos con análisis de layout, tablas y OCR; produce un `DoclingDocument` |
| **Motor de OCR** | Programa que reconoce el texto de una imagen (RapidOCR, EasyOCR, Tesseract, servicios de la nube) |
| **PDF mixto** | PDF con páginas con texto y páginas escaneadas |
| **Token** | Unidad en la que un modelo cuenta el texto; no es una palabra y depende del tokenizador |
| **Chunking híbrido (Docling)** | Fragmentar por jerarquía del documento y límite de tokens; **no** es la búsqueda híbrida |
| **`contextualize`** | Antepone los encabezados al texto de un fragmento: lo que conviene vectorizar |

## ➡️ Siguiente sesión
Ya sabes **recuperar bien**. El siguiente paso es **medir si el RAG responde bien y cuándo inventa**: evaluación de la generación, *groundedness* (fundamentación) y alucinaciones. También veremos **audio y video** como fuentes (se transcriben con Whisper).

---

**Implementación formal para estudiar:** `rag-playground/rag-intermedio` (repo `/home/oscar/Desktop/Personal/EDUTEK/GEN-AI/rag-playground`).